# I modelli del Deliverable 2, sulle quindici geometrie

I modelli sono quelli **consegnati**: Eq. (d2hier) per A, (d2phase) per C, (d2sites) per D1,
(d2move) per D2, la Gamma a link logaritmico per B, con i priori di `tab:bayesPriors` e le regole
di `tab:bayesDecisions`. Non e' cambiata una riga di specifica.

Quello che cambia e' **come** si campiona, che il Deliverable non vincola:

| leva | prima | qui | cambia la posteriore? |
|---|---|---|---|
| parametrizzazione | centrata | **non centrata** | **no**, e' la stessa distribuzione riscritta |
| metrica di massa | diagonale | **densa** su A e C | no, e' adattamento del campionatore |
| motore | PyMC/C | **numpyro** se c'e' JAX | no, stesso algoritmo NUTS |
| draws, tune, target\_accept | — | liberi | no |
| checkpoint | assente | **per blocco di catene** | no |

La distinzione va tenuta ferma perche' e' la prima cosa che verra' chiesta: una riparametrizzazione
non centrata e' una identita' algebrica, $\beta_c = \bar\beta + \tau z_c$ con $z_c \sim \mathcal
N(0,1)$, e produce esattamente la stessa distribuzione a posteriori di $\beta_c \sim \mathcal
N(\bar\beta, \tau)$. Cambia la geometria che il campionatore percorre, non l'oggetto campionato.

### Perche' A e C erano *provisional*, e cosa si puo' farci senza toccarle

Vale la pena essere precisi, perche' in una versione precedente di questo lavoro la diagnosi era
attribuita alla cosa sbagliata. Il difetto di Eq. (d2hier) **non** e' una deficienza di rango: la
risposta e' gia' un contrasto appaiato, quindi nel modello non c'e' nessun indicatore di lock da
confondere con la frequenza. Il difetto e' un altro, e sono due difetti insieme.

**Una cresta additiva fra tre raggruppamenti incrociati.** In
$\mu_i = \beta_{c[i]} + u_{k[i]} + u_{b[i]}$ si puo' aggiungere $\epsilon$ a tutti i $\beta_c$ e
toglierlo a tutti gli $u_k$ senza muovere un solo $\mu_i$. A trattenere la traslazione restano solo
i priori: $K$ termini $u_k$ pagano $K\epsilon^2/2\sigma_k^2$, e $\bar\beta$ paga il proprio
Student-$t_4(0,0.5)$. Quando $\sigma_k$ e' grande quel costo e' quasi nullo e la verosimiglianza e'
piatta lungo quella direzione. La firma e' **R-hat alto, ESS minuscolo, zero divergenze**: le
divergenze segnalano curvatura, e qui non ce n'e', c'e' una cresta.

**Un imbuto sulle scale.** $\beta_c \sim \mathcal N(\cdot, \tau)$ nella forma centrata, con quindici
geometrie e un priore Half-Student-$t_4$ a code pesanti su $\tau$, e' l'imbuto di Neal.

Le due cose si curano in modo diverso e nessuna delle due cure tocca il modello. L'imbuto si chiude
con la parametrizzazione non centrata. La cresta **non si chiude affatto**: e' una proprieta' del
modello come e' scritto, e resta una direzione poco determinata anche a campionamento perfetto. Cio'
che una metrica di massa densa fa e' permettere al campionatore di *percorrerla*, e quindi di
riportare una posteriore correttamente larga invece di diagnostiche rotte. Su questo il notebook e'
esplicito: se $\bar\beta$ resta largo, quella larghezza e' il risultato, non un fallimento.

### Tre errata sul testo, non sui modelli

1. **Le fasi sono dieci, non otto.** `tab:bayesPriors` dice "the eight per-phase offsets" e i dati
   raccolti hanno `phase_idx` da 0 a 9. Il modello resta "un offset per fetta di fase"; si fitta
   sulle dieci effettivamente spazzate e si corregge il numero nel testo. Con dieci gruppi invece di
   otto la deviazione standard a posteriori di $\log \sigma_\phi$ passa da circa $0{,}267$ a
   $0{,}236$: il cambio aiuta, non danneggia.
2. **Il segno di $M1$.** `tab:bayesDecisions` chiede $\Pr(\delta_O < 0) \ge 0.95$ e lo legge come
   "overlap reduces the deficit". Ma $\delta_O$ e' la pendenza su $d$, che e' **negativo** quando il
   lock recupera meno dei suoi controlli: ridurre il deficit vuol dire far salire $d$, cioe'
   $\delta_O > 0$. La regola contraddice la propria glossa. Si usa il segno corretto e lo si
   dichiara come erratum sulla regola. La probabilita' a priori resta $0{,}50$.
3. **$B$ non ha una soglia.** `tab:bayesDecisions` non contiene nessuna riga per
   $e^{\theta_{\mathrm{lock}}}$: le due righe di $H1$ sono entrambe scritte in $\bar\beta$, che e' il
   parametro di A. B ha quindi un estimando e una direzione prevista, e nessuna soglia. Qui la
   posteriore si riporta senza verdetto, e la mancanza si dichiara.

## 1, Ambiente e dipendenze

La cella 1a **non importa** numpy, pymc o arviz: se installa qualcosa, i moduli gia' in memoria
resterebbero quelli vecchi e il primo fit fallirebbe con un errore che non nomina la causa. Se
installa, il runtime si riavvia e basta rilanciare *Run all*.

In [ ]:
#@title 1a. Dipendenze e riavvio del runtime  { display-mode: "form" }
import sys, os, subprocess, importlib.metadata as _md

# Stessa specifica degli altri notebook del progetto: girano sullo stesso runtime, e se le
# specifiche divergessero ognuno reinstallerebbe pacchetti a danno dell'altro.
REQUISITI = [
    ("pymc",    "pymc>=5.20,<6",  lambda v: (5, 20) <= v < (6, 0, 0)),
    ("arviz",   "arviz==0.22.0",  lambda v: v == (0, 22, 0)),
    ("numpy",   "numpy>=1.26,<3", lambda v: (1, 26) <= v < (3, 0, 0)),
    ("pandas",  "pandas>=2,<4",   lambda v: (2, 0, 0) <= v < (4, 0, 0)),
    ("scipy",   "scipy>=1.11,<2", lambda v: (1, 11) <= v < (2, 0, 0)),
    ("pyarrow", "pyarrow",        lambda v: True),
    ("h5netcdf","h5netcdf",       lambda v: True),
    # h5py NON e' facoltativo anche se nessuna cella lo importa: h5netcdf e' solo la facciata,
    # e il backend che scrive davvero il file e' h5py. Senza, .to_netcdf() solleva
    #   ImportError: No module named 'h5py', backend not available
    # e -- questo e' il punto -- lo solleva DOPO il campionamento, quando il checkpoint si
    # scrive: si perderebbero ore di fit proprio nel gesto che doveva metterle al sicuro.
    ("h5py",    "h5py",           lambda v: True),
]
def _ver(nome):
    try: return tuple(int(x) for x in _md.version(nome).split(".")[:3] if x.isdigit())
    except Exception: return None

SPEC = [spec for _, spec, _ in REQUISITI]

def _controlla():
    """Quali requisiti sono fuori specifica adesso. La cella 2a lo richiama DOPO aver installato
    le dipendenze della pipeline: quelle tirano dentro torch e transformers, che possono
    spostare numpy sotto i piedi di un notebook gia' avviato."""
    fuori = []
    for nome, _spec, ok in REQUISITI:
        v = _ver(nome)
        if v is None or not ok(v): fuori.append(nome)
    return fuori

# Un file di vincoli: qualunque installazione successiva -- comprese le dipendenze del
# repository nella cella 2a -- non potra' spostare queste versioni.
CONSTRAINTS = ("/content/_patchaliasing_constraints.txt" if os.path.isdir("/content")
               else os.path.join(os.path.expanduser("~"), "_patchaliasing_constraints.txt"))
with open(CONSTRAINTS, "w") as _fh:
    _fh.write("\n".join(s for s in SPEC if any(c in s for c in "<>=")) + "\n")

_da_installare = []
for nome, spec, ok in REQUISITI:
    v = _ver(nome)
    if v is None or not ok(v):
        _da_installare.append(spec)
        print(f"  {nome}: {'assente' if v is None else '.'.join(map(str, v))} -> {spec}")

# numpyro e' FACOLTATIVO e cambia solo la velocita': stesso NUTS, stessa posteriore. Su 136 mila
# osservazioni di Student-t la differenza fra un fit di quaranta minuti e uno di due e' tutta qui.
USA_NUMPYRO = True   #@param {type:"boolean"}
if USA_NUMPYRO and _ver("numpyro") is None:
    _da_installare += ["numpyro"]
    print("  numpyro: assente -> numpyro (facoltativo, solo velocita')")

if _da_installare:
    print("\ninstallo:", " ".join(_da_installare))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_da_installare], check=True)
    print("\n" + "="*70)
    print("RUNTIME DA RIAVVIARE. Rilancia Runtime -> Run all: alla seconda passata")
    print("questa cella non installa piu' nulla e il notebook prosegue fino in fondo.")
    print("="*70)
    try:
        import IPython; IPython.Application.instance().kernel.do_shutdown(True)
    except Exception:
        pass
else:
    print("dipendenze a posto, nessun riavvio")

In [ ]:
#@title 1b. Import, costanti e soglie del Deliverable 2  { display-mode: "form" }
import os, sys, json, time, hashlib, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, pymc as pm, arviz as az
import pytensor.tensor as pt
warnings.filterwarnings("ignore")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 250)
TABELLE_TESTO = True    # stampa in testo: l'output ricco non arriva a ogni frontend

def mostra(df, dec=None):
    d = df.round(dec) if dec is not None else df
    print(d.to_string())

def banner(t):
    print("\n" + "="*96); print(t); print("="*96)

# --------------------------------------------------------------------- popolazione e disegno
FS        = 512.0
BANDA     = (2.0, 250.0)     # il Deliverable 2 dichiara [2,250] Hz
DELTA_F   = 1.0              # passo dello sweep, il floor di sigma_F in Eq. (d2move)
NU        = 4                # i gradi di liberta' dello Student-t, fissati dal Deliverable

POPOLAZIONE = [(8,8),(16,8),(16,12),(16,16),(24,8),(24,12),(24,16),(24,20),(24,24),
               (32,8),(32,12),(32,16),(32,20),(32,24),(32,32)]
TAG = {f"p{P}-s{S}" for P, S in POPOLAZIONE}

# --------------------------------------------------------------------- campionamento
SEED      = 42
CHAINS    = 4
BLOCCO    = 2      #@param {type:"integer"}   catene per checkpoint: 2 = si perde al piu' meta' fit
CORES     = min(BLOCCO, os.cpu_count() or 2)
DRAWS     = 1500   #@param {type:"integer"}
TUNE      = 1500   #@param {type:"integer"}
MODO_RAPIDO = False  #@param {type:"boolean"}   sottocampiona e accorcia: per provare, non per riportare

# --------------------------------------------------------------------- gate, dal Deliverable 2
RHAT_MAX, ESS_MIN, DIV_MAX = 1.01, 1000, 0
SOGLIA = 0.95                       # "supported at 0.95, refuted at 0.05"

# --------------------------------------------------------------------- soglie delle regole
LOG_08, LOG_11 = np.log(0.8), np.log(1.1)   # H1 comportamentale, H2
LOG_12         = np.log(1.2)                 # H1 rappresentazionale supportata
ROPE_KAPPA     = 0.10                        # |kappa_F - 1| < 0.1
SITI_MIN       = 10                          # condizione di identificazione di D2

if MODO_RAPIDO:
    DRAWS, TUNE = 300, 300
    print("*** MODO RAPIDO: draws e tune ridotti, dati sottocampionati. NON riportabile. ***")

print(f"numpy {np.__version__} | pandas {pd.__version__} | pymc {pm.__version__} | arviz {az.__version__}")
print(f"popolazione: {len(POPOLAZIONE)} geometrie   banda {BANDA[0]:.0f}-{BANDA[1]:.0f} Hz")
print(f"campionamento: {CHAINS} catene x {DRAWS} draws dopo {TUNE} tune, a blocchi di {BLOCCO}")
print(f"gate: R-hat <= {RHAT_MAX}, ESS bulk e tail >= {ESS_MIN}, divergenze <= {DIV_MAX}")

## 2, I dati

I quattro Parquet che il Deliverable nomina come input: `contrasts` per A e C, `mdl_cells` per B,
`collapse` per D1, `sites` per D2.

La **2a** li rigenera da zero dalla pipeline, una geometria per processo, ed e' facoltativa. La
**2b** li legge, e sa leggerli in entrambe le forme che la pipeline ha prodotto nel tempo: una
riga per tripletta, oppure una riga per braccio, da cui le triplette vengono ricostruite
applicando l'Eq. (d2contrast) invece di fidarsi di una colonna precalcolata.

In [ ]:
#@title 2a. RIGENERAZIONE DA ZERO della raccolta (facoltativa)  { display-mode: "form" }
# Rifa' i quattro Parquet dalla pipeline invece di leggerli. Serve quando la raccolta esistente
# viene da una versione diversa del collect, o quando si vuole una provenienza tracciata da capo.
#
# Due vincoli reali dell'ambiente decidono come e' scritta questa cella, e sono gia' stati pagati
# una volta. La MEMORIA: un unico processo che percorre le quindici geometrie viene ucciso dal
# runtime (exit -9, SIGKILL per RAM esaurita), perche' i tensori dei checkpoint gia' visitati non
# tornano al sistema; ogni geometria gira quindi in un PROCESSO SEPARATO che alla fine muore e
# libera tutto. Il TEMPO: su CPU la raccolta completa e' lunga, quindi gli shard si scrivono man
# mano e rieseguire la cella riparte dalla geometria dove si era fermata invece che da capo.
#
# Lasciare RIGENERA = False per usare una raccolta gia' pronta.
#
# QUESTA CELLA NON PUO' FERMARE IL NOTEBOOK. Rigenerare dipende da cose che stanno fuori dal
# notebook -- il branch, la rete, i checkpoint da scaricare, la RAM del runtime -- e nessuna di
# queste deve poter impedire di rifittare i modelli su una raccolta che c'e' gia'. Se qualcosa
# manca, la cella dice che cosa e dove l'ha cercato, e passa oltre: le celle da 2b in poi girano
# comunque. Una cella preparatoria che fa morire l'analisi e' un difetto della cella.
RIGENERA = False   #@param {type:"boolean"}
PIPE_SORGENTE = ""  #@param {type:"string"}   cartella con collect.py e probe_lib.py; vuoto = cercata

if not RIGENERA:
    print("Rigenerazione non richiesta: si legge una raccolta esistente (cella 2b).")

# Il corpo gira una volta sola, quando RIGENERA e' vero. E' scritto come ciclo perche' cosi' un
# controllo preliminare che non torna esce con 'break' invece di sollevare un'eccezione, che si
# porterebbe dietro tutto il resto del notebook.
for _passata_2a in ([0] if RIGENERA else []):
    #@title 2a. Clona la pipeline e installa le sue dipendenze  { display-mode: "form" }
    import shutil

    REPO_URL     = "https://github.com/FedericoSabbadini/patchAliasing.git"  #@param {type:"string"}
    REPO_REF     = "main"  #@param {type:"string"}
    GITHUB_TOKEN = ""  #@param {type:"string"}
    CLONE_DIR    = "/content/patchAliasing"  #@param {type:"string"}

    def _sanitize(t): return t.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else t

    COMMIT = "(nessun clone)"
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    if os.path.isdir(CLONE_DIR): shutil.rmtree(CLONE_DIR)
    print(f"clone di {REPO_URL} @ {REPO_REF} ...")
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, url, CLONE_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone fallito:\n" + _sanitize(r.stderr.strip()))
        print("Se il repository e' privato serve un token di lettura in GITHUB_TOKEN. Si cerca\n"
              "comunque la pipeline altrove: PIPE_SORGENTE, Drive, cartella di lavoro.")
    else:
        subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", REPO_URL], check=False)
        COMMIT = subprocess.run(["git", "-C", CLONE_DIR, "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip()
        print("commit:", COMMIT)

    # --- Dove sta la pipeline -------------------------------------------------------------
    # Il repository e' stato riorganizzato: i quattro moduli stanno in chronos/bayesian/
    # SUPPORT_SCRIPTS, non piu' direttamente in chronos/bayesian. Era questa, e solo questa,
    # la causa del "collect.py manca in /content/patchAliasing/chronos/bayesian": i file
    # c'erano, la cella guardava nella cartella di prima. Si guarda quindi in entrambi i posti,
    # e in piu' dove l'utente puo' averli messi a mano; e quando non si trova niente si stampa
    # l'elenco delle cartelle guardate, invece del nome di un file e basta.
    RICHIESTI = ["collect.py", "probe_lib.py", "model_loader.py", "checkpointing.py"]

    def _e_pipeline(d):
        return d and all(os.path.isfile(os.path.join(str(d), f)) for f in RICHIESTI)

    _cand = []
    if PIPE_SORGENTE: _cand.append(PIPE_SORGENTE)
    for _base in (CLONE_DIR, "/content/drive/MyDrive/patchAliasing",
                  str(globals().get("RADICE") or ""), str(Path.cwd())):
        if not _base: continue
        _cand += [os.path.join(_base, "chronos", "bayesian", "support_scripts"),
                  os.path.join(_base, "chronos", "bayesian")]
    _cand += [str(Path.cwd()), "/content/drive/MyDrive/patchAliasing/support_scripts"]
    _cand = list(dict.fromkeys(_cand))          # senza doppioni, nell'ordine di preferenza

    PIPE_DIR = next((c for c in _cand if _e_pipeline(c)), None)
    if PIPE_DIR is None:
        print("\n" + "!"*78)
        print("PIPELINE NON TROVATA: la rigenerazione non parte, il resto del notebook si'.")
        print("Servono, nella stessa cartella: " + ", ".join(RICHIESTI))
        print("\nCartelle guardate:")
        for c in _cand:
            _m = ([f for f in RICHIESTI if not os.path.isfile(os.path.join(str(c), f))]
                  if os.path.isdir(str(c)) else ["(la cartella non esiste)"])
            print(f"   {c}\n      manca: {', '.join(_m)}")
        print("\nDue vie:")
        print(f"  - fare push di chronos/bayesian/support_scripts sul branch '{REPO_REF}' e")
        print("    rieseguire questa cella (il clone viene rifatto);")
        print("  - oppure copiare quella cartella su Drive e scriverne il percorso in "
              "PIPE_SORGENTE.")
        print("Intanto le celle 2b e successive leggono la raccolta gia' presente.")
        print("!"*78 + "\n")
        break
    print("pipeline:", PIPE_DIR)

    _rc = subprocess.run([sys.executable, "-c", "import checkpointing, model_loader"],
                         cwd=PIPE_DIR, capture_output=True, text=True)
    if _rc.returncode != 0:
        print("\nI moduli ci sono ma non si importano:\n" + _rc.stderr.strip())
        print("La rigenerazione si ferma qui; le celle 2b e successive girano.")
        break

    # Le dipendenze si LEGGONO dal pyproject.toml del repository, ma il pacchetto non si installa:
    # il repository e' una raccolta di script senza [build-system], quindi 'pip install -e' fallisce
    # con "Getting requirements to build editable did not run successfully". Prendere l'elenco e
    # installarlo direttamente ottiene lo stesso risultato senza passare dal build backend.
    try:
        import tomllib
        with open(os.path.join(CLONE_DIR, "pyproject.toml"), "rb") as fh:
            deps = tomllib.load(fh)["project"]["dependencies"]
    except Exception as e:
        print("pyproject non leggibile, uso l'elenco di riserva:", e)
        deps = ["chronos-forecasting>=2.2.2", "transformers>=4.57.6", "accelerate>=1.13.0",
                "huggingface-hub>=0.36.2", "datasets>=5.0.0", "scikit-learn>=1.8.0", "tqdm"]
    SKIP = {"black", "isort", "matplotlib", "pillow"}      # strumenti di sviluppo, non servono qui
    deps = [d for d in deps
            if d.split(">=")[0].split("==")[0].split("<")[0].strip().lower() not in SKIP]
    print("\ndipendenze dal repository:", ", ".join(deps))
    _pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", CONSTRAINTS, *deps],
                          capture_output=True, text=True)
    if _pip.returncode != 0:
        print("\n" + "!"*78)
        print("installazione delle dipendenze della pipeline fallita:")
        print((_pip.stderr or _pip.stdout).strip()[-3000:])
        print("Senza queste la raccolta non parte; le celle 2b e successive girano comunque.")
        print("!"*78)
        break

    # Le dipendenze della pipeline tirano dentro torch e transformers, che possono spostare
    # numpy o pymc. Se e' successo, si rimettono a posto e si riavvia: le versioni caricate in
    # memoria sono ormai quelle vecchie, e proseguire darebbe errori che sembrano di modello.
    rotte = _controlla()
    if rotte:
        _fix = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *SPEC],
                              capture_output=True, text=True)
        if _fix.returncode != 0:
            print("\n" + "!"*78)
            print("Le dipendenze della pipeline hanno spostato " + ", ".join(rotte) +
                  ", e il ripristino non e' riuscito:")
            print((_fix.stderr or _fix.stdout).strip()[-2000:])
            print("Riavviare il runtime e rilanciare Run all con RIGENERA = False.")
            print("!"*78)
            break
        print("\nLe versioni condivise erano state spostate e sono state ripristinate.")
        print("Riavvio: quando il kernel torna su, rilanciare Runtime -> Run all.")
        os.kill(os.getpid(), 9)
    print("\npipeline pronta in", PIPE_DIR)

    #@title 2b. Rigenera i Parquet, una geometria per processo  { display-mode: "form" }
    POP_PIPELINE = "deliverable3"  #@param ["deliverable3", "extended21", "all22"]
    REGEN_OUT  = "/content/drive/MyDrive/patchAliasing/raccolta_D2/data"  #@param {type:"string"}
    BATCH_SIZE = 8  #@param {type:"integer"}
    SALTA_PREFLIGHT = False  #@param {type:"boolean"}
    RIPARTI_DA_ZERO = False  #@param {type:"boolean"}
    MEMORIA_LIMITATA = True  #@param {type:"boolean"}
    SITI_PER_BLOCCO  = 4  #@param {type:"integer"}

    if REGEN_OUT.startswith("/content/drive/"):
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
        except Exception as e:
            REGEN_OUT = "/content/patchaliasing_data"
            print("Drive non disponibile, scrivo sul runtime:", REGEN_OUT, f"({e})")
    LOG_DIR = os.path.join(os.path.dirname(REGEN_OUT.rstrip("/")), "log")
    os.makedirs(LOG_DIR, exist_ok=True)

    def run_step(cmd, cwd, log_path):
        # L'output del figlio viene SEMPRE catturato e salvato. Senza questo si vede solo il
        # codice di uscita e il traceback vero resta invisibile: era il difetto della versione
        # precedente, che costringeva a indovinare la causa a ogni tentativo.
        righe = []
        with open(log_path, "w") as fh:
            proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                fh.write(line); fh.flush(); righe.append(line)
                print(line, end="", flush=True)
            rc = proc.wait()
        return rc, righe

    def spiega(rc, righe):
        testo = "".join(righe)
        if rc == -9:
            return ("il runtime ha ucciso il processo per RAM esaurita (SIGKILL).\n"
                    f"  BATCH_SIZE ({BATCH_SIZE}) NON e' la leva: governa il passaggio in avanti,\n"
                    "  che avviene dopo che i bracci sono gia' tutti in memoria.\n"
                    f"  Abbassare SITI_PER_BLOCCO (ora {SITI_PER_BLOCCO}) a 2 o 1, oppure usare un\n"
                    "  runtime High-RAM. Le geometrie gia' complete non vengono perse.")
        for frammento, diagnosi in [
            ("use a new output directory",
             "la cartella di output contiene i resti di una esecuzione interrotta.\n"
             "  Questa cella la svuota da sola: se l'errore persiste, cambiare REGEN_OUT."),
            ("does not match this design",
             "il manifest viene da una configurazione diversa della pipeline.\n"
             "  Cambiare REGEN_OUT, oppure cancellarla a mano."),
            ("unrecognized arguments: --population",
             "il collect.py clonato e' la versione PRECEDENTE, senza --population.\n"
             "  Fare push dei file patchati sul branch e rieseguire la cella: il clone si rifa'."),
            ("PATCHALIASING_POPULATION",
             "probe_lib.py clonato non conosce la popolazione richiesta: push mancante."),
            ("CUDA", "la pipeline ha cercato la GPU: verificare che --device cpu sia passato."),
            ("HTTPError", "download da Hugging Face fallito: rete o repository dei checkpoint."),
            ("OSError", "problema di I/O o di accesso ai checkpoint."),
            ("ModuleNotFoundError",
             "manca un pacchetto: rieseguire la cella 2a, che installa le dipendenze del repo."),
            ("No space left", "disco pieno: liberare spazio o cambiare REGEN_OUT."),
        ]:
            if frammento in testo: return diagnosi
        return "causa non riconosciuta: il traceback completo e' qui sotto."

    GUASTO = []      # un guasto si registra e si stampa; non solleva
    def fallisci(tag, rc, righe, log_path):
        coda = "".join(righe[-150:]).rstrip() or "(il processo non ha prodotto output)"
        GUASTO.append(tag)
        print("\n" + "!"*78)
        print(f"{tag}: collect.py e' uscito con codice {rc}")
        print(f"CAUSA: {spiega(rc, righe)}")
        print(f"log completo: {log_path}")
        print(f"--- ultime righe ---\n{coda}")
        print("La rigenerazione si ferma qui. Le geometrie gia' complete restano in REGEN_OUT,")
        print("e rieseguire questa cella riparte da dove si era arrivati.")
        print("Intanto le celle 2b e successive leggono la raccolta gia' presente.")
        print("!"*78)

    # --- Driver a memoria limitata ------------------------------------------------------
    # collect_contrasts costruisce TUTTI i bracci della geometria e poi chiama probe.measure
    # una volta sola. Per una geometria con molti siti sono centinaia di migliaia di finestre
    # vive insieme, e il runtime uccide il processo (SIGKILL, exit -9). BATCH_SIZE non c'entra:
    # governa solo il passaggio in avanti, che avviene DOPO la materializzazione.
    #
    # In probe_lib ogni operazione e' riga per riga: _batches affetta, forecast concatena per
    # blocchi, measure cicla per indice. Non c'e' accoppiamento fra bracci, quindi elaborare i
    # siti a blocchi da' lo STESSO risultato e nello stesso ordine -- il ciclo sui generatori
    # resta esterno e i siti restano nel loro ordine. Cambia solo quando si alloca memoria.
    #
    # Il file del repository non viene modificato: il driver lo importa, sostituisce la sola
    # funzione e chiama collect.main. L'isolamento per geometria resta.
    DRIVER_SRC = '\nimport os, sys, gc, resource\nimport numpy as np, pandas as pd\nimport probe_lib as pl\nimport collect as C\n\nCHUNK = int(os.environ.get("SITI_PER_BLOCCO", "4"))\nCOLS = ["model","P","S","overlap","generator","bg_id","f_lock","family","cpp","delta",\n        "phase_idx","phase","role","f","is_lock","R","dphase","f_hat","h","h_truth"]\n\ndef _rss():\n    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024**2\n\ndef collect_contrasts_lowmem(probe, cfg):\n    P, S = probe.P, probe.S\n    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}\n    sites = [f for f, d in offsets.items() if np.isfinite(d)]\n    print("    siti utili: %d, a blocchi di %d" % (len(sites), CHUNK), flush=True)\n    frames = []\n    for gen in cfg.generators:\n        pool = pl.background_pool(gen, cfg.n_bg, pl.CTX + pl.PRED)\n        for c0 in range(0, len(sites), CHUNK):\n            blocco = sites[c0:c0+CHUNK]\n            contexts, futures, freqs, meta = [], [], [], []\n            for fk in blocco:\n                phases = pl.phases_Sf(fk, cfg.n_phase_contrast)\n                for bg_id, bg in enumerate(pool):\n                    for ph_idx, ph in enumerate(phases):\n                        d_fk = offsets[fk]\n                        for role, f in (("lock", fk), ("lo", fk - d_fk), ("hi", fk + d_fk)):\n                            full = pl.build_context(bg, f, ph, pl.CTX + pl.PRED)\n                            contexts.append(np.array(full[:pl.CTX]))\n                            futures.append(np.array(full[pl.CTX:]))\n                            freqs.append(f)\n                            meta.append(dict(generator=gen, bg_id=bg_id, f_lock=fk, delta=d_fk,\n                                             phase_idx=ph_idx, phase=float(ph), role=role,\n                                             f=float(f)))\n            if not contexts:\n                continue\n            R, dphase, f_hat, f_hat_truth = probe.measure(\n                np.stack(contexts), np.stack(futures), np.array(freqs), k=cfg.fhat_topk)\n            out = pd.DataFrame(meta)\n            out["R"] = R\n            out["dphase"] = dphase\n            out["f_hat"] = f_hat[:, 0]\n            out["h"] = pl.localisation_hit(f_hat, out["f"].to_numpy(float), tol=cfg.fhat_tol_hz)\n            out["h_truth"] = pl.localisation_hit(f_hat_truth, out["f"].to_numpy(float),\n                                                 tol=cfg.fhat_tol_hz)\n            frames.append(out)\n            del contexts, futures, freqs, meta, R, dphase, f_hat, f_hat_truth\n            gc.collect()\n            print("    %s siti %d-%d/%d  righe finora %d  RSS max %.1f GB"\n                  % (gen, c0+1, c0+len(blocco), len(sites),\n                     sum(len(x) for x in frames), _rss()), flush=True)\n    if not frames:\n        return pd.DataFrame()\n    out = pd.concat(frames, ignore_index=True)\n    out["is_lock"] = (out["role"] == "lock").astype(np.int8)\n    out["model"] = probe.tag\n    out["P"], out["S"] = P, S\n    out["overlap"] = (P - S) / P\n    out["cpp"] = out["f_lock"] * P / pl.FS\n    out["family"] = [pl.lock_family(f, P, S) for f in out["f_lock"]]\n    return out[COLS]\n\nC.collect_contrasts = collect_contrasts_lowmem\nprint("driver a memoria limitata attivo (blocchi da %d siti)" % CHUNK, flush=True)\nsys.exit(C.main(sys.argv[1:]))\n'
    # La popolazione si imposta nell'AMBIENTE, non solo come flag: probe_lib la legge all'import,
    # e se il flag non coincide con quella gia' caricata collect.py riavvia il processo. Un
    # riavvio ripartirebbe dallo script d'ingresso, e in ogni caso e' lavoro inutile: cosi' i
    # figli la ereditano e nessun riavvio serve.
    os.environ['PATCHALIASING_POPULATION'] = POP_PIPELINE
    ENTRY = 'collect.py'
    if MEMORIA_LIMITATA:
        ENTRY = '_collect_lowmem.py'
        with open(os.path.join(PIPE_DIR, ENTRY), 'w') as fh: fh.write(DRIVER_SRC)
        os.environ['SITI_PER_BLOCCO'] = str(SITI_PER_BLOCCO)
        print(f'punto di ingresso: {ENTRY}  (blocchi da {SITI_PER_BLOCCO} siti)')
    else:
        print('punto di ingresso: collect.py (nessuna limitazione di memoria)')

    def _ram_gb():
        try:
            import psutil; return psutil.virtual_memory().available/2**30
        except Exception:
            return float("nan")

    # --- Il repository clonato contiene davvero i file patchati? ------------------------
    # Controllo prima di qualunque raccolta: se il push non e' arrivato su main, o se il clone
    # lo precede, la pipeline non conosce --population e lo si scopre solo a meta' preflight
    # con un messaggio di argparse che non dice cosa fare.
    _src_collect = open(os.path.join(PIPE_DIR, "collect.py")).read()
    _src_probe   = open(os.path.join(PIPE_DIR, "probe_lib.py")).read()
    _manca = []
    if '"--population"' not in _src_collect: _manca.append("collect.py: flag --population")
    if "PATCHALIASING_POPULATION" not in _src_probe: _manca.append("probe_lib.py: popolazione selezionabile")
    FLAG_POP = [] if _manca else ["--population", POP_PIPELINE]
    if _manca:
        print("!"*74)
        print(f"La pipeline trovata ({PIPE_DIR}) NON ha le modifiche attese:")
        for m in _manca: print("   -", m)
        print("Il push dei file patchati non e' su questo branch, oppure il clone lo precede.")
        print("!"*74)
        if POP_PIPELINE != "deliverable3":
            print(f"POP_PIPELINE = '{POP_PIPELINE}' richiede la pipeline patchata, che qui non c'e'.")
            print("  - fare push di probe_lib.py e collect.py sul branch "
                  f"'{REPO_REF}', poi rieseguire questa cella (il clone viene rifatto);")
            print("  - oppure mettere POP_PIPELINE = 'deliverable3' per restare alle 15.")
            print("La rigenerazione si ferma qui; il resto del notebook gira.")
            break
        print("Si prosegue con le 15: e' l'unica popolazione che questa pipeline conosce.\n")
    else:
        print(f"pipeline patchata presente (popolazione '{POP_PIPELINE}')\n")

    # Preflight: una raccolta in miniatura su una cartella usa-e-getta. Dura un minuto e fa
    # emergere subito i problemi di ambiente -- pacchetti, accesso ai checkpoint, rete --
    # invece di scoprirli dopo ore. Se questo passa, il resto e' una questione di tempo e RAM.
    if not SALTA_PREFLIGHT:
        SMOKE_DIR = "/content/_preflight_smoke" if os.path.isdir("/content") else "./_preflight_smoke"
        shutil.rmtree(SMOKE_DIR, ignore_errors=True)
        print("="*70); print("PREFLIGHT: raccolta in miniatura su una geometria"); print("="*70)
        rc, righe = run_step([sys.executable, ENTRY, "--out", SMOKE_DIR, "--device", "cpu",
                              "--batch-size", str(BATCH_SIZE), "--smoke", "--models", "p8-s8",
                              *FLAG_POP],
                             PIPE_DIR, os.path.join(LOG_DIR, "preflight.log"))
        if rc != 0:
            fallisci("preflight", rc, righe, os.path.join(LOG_DIR, "preflight.log"))
        shutil.rmtree(SMOKE_DIR, ignore_errors=True)
        print("\nPREFLIGHT SUPERATO: ambiente, checkpoint e pipeline funzionano.\n")

    # --- Riprendere o ripartire: la decisione si prende guardando la cartella ------------
    # collect.py sa gia' riprendere: collect_model salta una geometria i cui shard risultano
    # validati dal manifest, senza nemmeno caricare il checkpoint. Quello che NON tollera e'
    # una cartella incoerente, e lo dice ("use a new output directory"). Un processo ucciso
    # a meta' puo' lasciarne due tipi:
    #   - l'archivio dei segnali scritto ma non registrato nel manifest -> l'intera cartella
    #     e' inservibile e va rifatta (e' l'errore che si era presentato);
    #   - uno shard su disco ma non nel manifest -> riguarda UNA geometria, che si rifa'
    #     con --force senza perdere le altre.
    # Qui si distingue fra i due casi invece di cancellare sempre tutto.
    # --- Si raccoglie tutto, sempre, da zero ---------------------------------------------
    # Niente migrazione, niente riuso di raccolte precedenti: il manifest porta l'impronta del
    # registro e delle sorgenti, e ogni tentativo di far combaciare due raccolte fatte con
    # codice o popolazioni diverse costa piu' tempo di quanto ne faccia risparmiare.
    # La cartella di destinazione viene svuotata alla prima esecuzione. Se il processo viene
    # interrotto, rieseguire questa cella riprende dalle geometrie mancanti: le shard sono
    # scritte in modo atomico e collect.py salta da solo quelle gia' validate dal manifest.
    _POP_EXTRA = {"deliverable3": set(),
                  "extended21": {(8,4),(8,5),(16,4),(16,15),(24,15),(32,15)},
                  "all22": {(8,4),(8,5),(16,4),(16,15),(24,15),(32,15),(32,28)}}[POP_PIPELINE]
    # le quindici sono POPOLAZIONE, dalla cella 1b: la rigenerazione non tiene una sua lista,
    # altrimenti le due potrebbero divergere senza che nessuno se ne accorga.
    POP_GEOMS = set(POPOLAZIONE) | _POP_EXTRA
    TABLES = ("contrasts", "mdl_cells", "mdl_bandtasks", "collapse")
    TAGS = [f"p{P}-s{S}" for P, S in sorted(POP_GEOMS)]
    print(f"popolazione '{POP_PIPELINE}': {len(TAGS)} geometrie")
    if _POP_EXTRA:
        print("  oltre alle 15 della Tabella 1:",
              ", ".join(f"p{P}-s{S}" for P, S in sorted(_POP_EXTRA)))

    def _quante_complete(d):
        mp = os.path.join(d, "collection_manifest.json")
        if not os.path.isfile(mp):
            return 0
        try:
            sh = json.load(open(mp)).get("shards", {})
        except Exception:
            return 0
        n = 0
        for tag in TAGS:
            reg = [t for t in TABLES if f"{t}__{tag}" in sh]
            if reg and all(os.path.isfile(os.path.join(d, "raw", f"{t}__{tag}.parquet"))
                           for t in reg):
                n += 1
        return n

    def _svuota(motivo):
        print(f"svuoto {REGEN_OUT}: {motivo}")
        if os.path.isdir(REGEN_OUT):
            shutil.rmtree(REGEN_OUT)
        os.makedirs(REGEN_OUT, exist_ok=True)

    _gia = _quante_complete(REGEN_OUT) if os.path.isdir(REGEN_OUT) else 0
    if RIPARTI_DA_ZERO:
        _svuota("RIPARTI_DA_ZERO attivo")
        _gia = 0
    elif _gia == 0 and os.path.isdir(REGEN_OUT) and os.listdir(REGEN_OUT):
        # c'e' qualcosa ma nessuna geometria completa: resti di un tentativo precedente
        _svuota("nessuna geometria completa, sono resti di un tentativo interrotto")
    os.makedirs(REGEN_OUT, exist_ok=True)
    print(f"\nda raccogliere: {len(TAGS)} geometrie   gia' complete qui: {_gia}")

    t_start = time.time()
    _ripulito = False
    i = 0
    while i < len(TAGS):
        tag = TAGS[i]
        log_path = os.path.join(LOG_DIR, f"{tag}.log")
        print(f"\n{'='*70}\n[{i+1}/{len(TAGS)}] {tag}   RAM libera {_ram_gb():.1f} GB   "
              f"trascorso {(time.time()-t_start)/60:.0f} min\n{'='*70}", flush=True)
        cmd = [sys.executable, ENTRY, "--out", REGEN_OUT, "--device", "cpu",
               "--batch-size", str(BATCH_SIZE), "--models", tag, *FLAG_POP]
        rc, righe = run_step(cmd, PIPE_DIR, log_path)
        if rc != 0:
            testo = "".join(righe)
            # Due guasti si riparano da soli, una volta ciascuno. Non e' indulgenza verso
            # l'errore: sono gli unici due stati che una cartella puo' assumere dopo
            # un'interruzione, e la pipeline stessa dice come uscirne.
            if "does not match this design" in testo and not _ripulito:
                _svuota("il manifest presente viene da un'altra configurazione")
                _ripulito = True
                i = 0                      # si ricomincia dalla prima geometria
                continue
            if "untracked shard" in testo:
                print(f"  {tag}: shard non registrata, la rifaccio con --force", flush=True)
                rc, righe = run_step(cmd + ["--force"], PIPE_DIR, log_path)
            if rc != 0:
                fallisci(tag, rc, righe, log_path)
                break
        i += 1
    if GUASTO:
        break

    print(f"\n{'='*70}\nmerge finale sulle {len(TAGS)} geometrie\n{'='*70}", flush=True)
    log_path = os.path.join(LOG_DIR, "merge.log")
    rc, righe = run_step([sys.executable, "collect.py", "--out", REGEN_OUT, "--merge-only",
                          *FLAG_POP],
                         PIPE_DIR, log_path)
    if rc != 0:
        fallisci("merge", rc, righe, log_path)
        break

    DATA_DIR = REGEN_OUT
    mancanti = [f for f in ["02_contrasts.parquet", "02_collapse.parquet"]
                if not os.path.isfile(os.path.join(DATA_DIR, f))]
    if mancanti:
        print(f"la raccolta e' terminata ma mancano {mancanti} in {DATA_DIR}: la cella 2b")
        print("usera' la raccolta gia' presente.")
        break
    print(f"\nrigenerata in {DATA_DIR}")
    print(f"commit {COMMIT}   tempo totale {(time.time()-t_start)/60:.0f} min")
    print(f"log per geometria in {LOG_DIR}")

    # la raccolta appena prodotta diventa l'input della cella 2b
    DATA_DIR_RIGENERATA = REGEN_OUT
    print(f"\nraccolta rigenerata in {REGEN_OUT}")
    print("La cella 2b la usera' come DATA_DIR senza che tu debba impostarlo.")

In [ ]:
#@title 2b. Repo, percorsi e caricamento dei quattro Parquet  { display-mode: "form" }
MONTA_DRIVE  = True   #@param {type:"boolean"}
CLONA_REPO   = True   #@param {type:"boolean"}
REPO_URL     = "https://github.com/FedericoSabbadini/patchAliasing.git"  #@param {type:"string"}
REPO_REF     = "main"  #@param {type:"string"}
GITHUB_TOKEN = ""     #@param {type:"string"}
CLONE_DIR    = "/content/patchAliasing"  #@param {type:"string"}
DATA_DIR     = ""     #@param {type:"string"}   vuoto = cercato nel repo e su Drive
OUT_DIR      = "_run/modelli_D2"  #@param {type:"string"}

import sys, shutil, subprocess

# ------------------------------------------------------------------ Drive e repo
DRIVE = None
if MONTA_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    try:
        from google.colab import drive as _drv
        _drv.mount("/content/drive")
    except Exception as _e:
        print(f"Drive non montato ({type(_e).__name__}): si prosegue senza.")
if Path("/content/drive/MyDrive").is_dir():
    DRIVE = Path("/content/drive/MyDrive"); print(f"Drive: {DRIVE}")

# Il marcatore che identifica la radice del repository. Vanno tenuti ENTRAMBI i percorsi:
# probe_lib.py e' stato spostato in chronos/bayesian/support_scripts, e un clone piu' vecchio
# -- o una copia su Drive fatta prima dello spostamento -- ce l'ha ancora nel posto di prima.
MARCATORI = [Path("chronos/bayesian/support_scripts/probe_lib.py"),
             Path("chronos/bayesian/probe_lib.py")]
def _marcato(b):
    return any((Path(b) / m).is_file() for m in MARCATORI)

def _radice():
    if _marcato(CLONE_DIR): return Path(CLONE_DIR)
    for b in [Path.cwd(), *Path.cwd().parents]:
        if _marcato(b): return b
    if DRIVE is not None:
        for c in (DRIVE / "patchAliasing", DRIVE / "Colab Notebooks/patchAliasing"):
            if _marcato(c): return c
    return None

RADICE = _radice()
if RADICE is None and CLONA_REPO:
    _u = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    print(f"clono {REPO_URL} @ {REPO_REF} in {CLONE_DIR}")
    if os.path.isdir(CLONE_DIR): shutil.rmtree(CLONE_DIR)
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, _u, CLONE_DIR],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        _e = _r.stderr.strip().replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else _r.stderr.strip()
        # Il repository serve solo per trovare i Parquet: se DATA_DIR e' gia' indicato, o se i
        # dati stanno su Drive, il clone non e' necessario e non deve fermare niente.
        print("git clone fallito:\n" + _e +
              "\nSe il repository e' privato, metti un token in GITHUB_TOKEN. Si cercano\n"
              "comunque i Parquet su Drive, in DATA_DIR e nella cartella di lavoro.")
    RADICE = _radice()
if RADICE is not None:
    print(f"repo: {RADICE}")
    for _b in [str((RADICE / "chronos/bayesian/support_scripts").resolve()),
               str((RADICE / "chronos/bayesian").resolve())]:
        if os.path.isdir(_b) and _b not in sys.path: sys.path.insert(0, _b)

# ------------------------------------------------------------------ i quattro Parquet
ATTESI = ["02_contrasts.parquet", "02_mdl_cells.parquet",
          "02_collapse.parquet", "02_sites.parquet"]

def _trova_dati():
    cand = []
    # se la 2a ha appena rigenerato, quella raccolta ha la precedenza su tutto
    if globals().get("DATA_DIR_RIGENERATA"): cand.append(Path(DATA_DIR_RIGENERATA))
    if DATA_DIR: cand.append(Path(DATA_DIR))
    if RADICE is not None:
        # results/ contiene piu' raccolte: _run_clean_15 e' quella delle quindici geometrie del
        # Deliverable, _run_full_21 ne ha ventuno. La glob da sola le restituisce in ordine
        # arbitrario, e prendere la sbagliata non darebbe errore: le sei geometrie in piu'
        # verrebbero semplicemente scartate piu' sotto, in silenzio. Quindi si ordina.
        _run = sorted((RADICE / "chronos/bayesian/results").glob("*/data"),
                      key=lambda p: (0 if "clean_15" in p.parent.name else
                                     1 if "15" in p.parent.name else 2, p.parent.name))
        cand += _run
        cand += [RADICE / "chronos/bayesian/_run/full"]
    if DRIVE is not None:
        cand += [DRIVE / "patchAliasing/full/rigenerata_all22/data",
                 DRIVE / "patchAliasing/data", DRIVE / "patchAliasing"]
        cand += list((DRIVE / "patchAliasing").glob("*/data")) if (DRIVE/"patchAliasing").is_dir() else []
    cand += [Path.cwd(), Path.cwd() / "data", Path("/content/data")]
    for c in cand:
        if c.is_dir() and all((c / f).is_file() for f in ATTESI):
            return c
    # ultima risorsa: cerca 02_contrasts.parquet e prendi la sua cartella
    for radice in [p for p in (RADICE, DRIVE, Path("/content"), Path.cwd()) if p is not None]:
        try: t = next(Path(radice).rglob("02_contrasts.parquet"), None)
        except (OSError, PermissionError): t = None
        if t is not None and all((t.parent / f).is_file() for f in ATTESI):
            return t.parent
    raise FileNotFoundError(
        "I quattro Parquet non si trovano: " + ", ".join(ATTESI) +
        "\n   Cercati in: " + ", ".join(str(c) for c in cand[:8]) + " ..."
        "\n   Imposta DATA_DIR al percorso della cartella che li contiene.")

DATI = _trova_dati()
print(f"dati: {DATI}")

# ------------------------------------------------------------------ dove scrivere
if not os.path.isabs(OUT_DIR):
    _coda = OUT_DIR.lstrip("./")
    if DRIVE is not None:                       OUT_DIR = str(DRIVE / "patchAliasing" / _coda)
    elif RADICE is not None and not str(RADICE).startswith("/content"):
                                                OUT_DIR = str(RADICE / "chronos/bayesian" / _coda)
    else:                                       OUT_DIR = str(Path.cwd() / _coda)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
(Path(OUT_DIR) / "ckpt").mkdir(exist_ok=True)
print(f"risultati e checkpoint: {OUT_DIR}")
if OUT_DIR.startswith("/content/") and not OUT_DIR.startswith("/content/drive/"):
    print("  ATTENZIONE: sul runtime, non su Drive. Se la sessione salta si perdono i checkpoint.")

# ------------------------------------------------------------------ carica e filtra
CONTR = pd.read_parquet(DATI / "02_contrasts.parquet")
MDL   = pd.read_parquet(DATI / "02_mdl_cells.parquet")
COLL  = pd.read_parquet(DATI / "02_collapse.parquet")
SITI  = pd.read_parquet(DATI / "02_sites.parquet")
for _n, _d in (("contrasts", CONTR), ("mdl_cells", MDL), ("collapse", COLL), ("sites", SITI)):
    _fuori = sorted(set(_d.model.unique()) - TAG)
    _d.drop(_d.index[~_d.model.isin(TAG)], inplace=True)
    print(f"{_n:11s} {len(_d):8,d} righe   {_d.model.nunique():2d} geometrie"
          + (f"   scartate {len(_fuori)} fuori popolazione" if _fuori else ""))

assert CONTR.model.nunique() == len(POPOLAZIONE), \
    f"attese {len(POPOLAZIONE)} geometrie in contrasts, trovate {CONTR.model.nunique()}"

# ------------------------------------------------------------------ schema
# La raccolta puo' emettere i contrasti in due forme, e vanno trattate diversamente.
#
#   LARGA  una riga per tripletta, con R_lock, R_lo, R_hi (ed eventualmente d gia' calcolato)
#   LUNGA  una riga per BRACCIO, con role/is_lock e una sola colonna R: tre righe per tripletta
#
# Nella forma lunga le triplette vanno ricostruite, e il Deliverable dice esattamente come:
#   d = log(R_k + 0.01) - 1/2 [ log(R_- + 0.01) + log(R_+ + 0.01) ]        Eq. (d2contrast)
# "each lock f_k with its two controls, all three sharing the same background realisation and
# the same phase". La chiave della tripletta deve quindi includere il GENERATORE oltre a
# geometria, sfondo, armonica e fase: senza, le due famiglie di segnale finiscono nello stesso
# gruppo e la tripletta ne raccoglie quattro controlli invece di due.
EPS_D = 0.01

def _col(df, *nomi):
    for n in nomi:
        if n in df.columns: return n
    return None

print("\ncolonne di contrasts:")
print("   " + ", ".join(CONTR.columns))

def _log1(x):
    return np.log(np.clip(np.asarray(x, float), 0.0, None) + EPS_D)

_rl = _col(CONTR, "R_lock", "R_k", "recovery_lock")
_lo = _col(CONTR, "R_lo", "R_minus", "R_ctrl_lo")
_hi = _col(CONTR, "R_hi", "R_plus", "R_ctrl_hi")
_r1 = _col(CONTR, "R", "recovery")
_ruolo = _col(CONTR, "is_lock", "role", "is_locked")
FORMA = "larga" if (_rl and _lo and _hi) else ("lunga" if (_r1 and _ruolo) else "ignota")
print(f"forma dei contrasti: {FORMA.upper()}")

if FORMA == "lunga":
    _chiave = [c for c in ("model", "generator", "bg_id", "f_lock", "phase_idx")
               if c in CONTR.columns]
    print(f"   chiave della tripletta: {', '.join(_chiave)}")
    _lk = (CONTR[_ruolo].astype(bool) if CONTR[_ruolo].dtype != object
           else CONTR[_ruolo].astype(str).str.contains("lock", case=False))
    _porta = [c for c in ("P", "S", "overlap", "family") if c in CONTR.columns
              and c not in _chiave]
    _A = (CONTR[_lk].groupby(_chiave, as_index=False)
                    .agg(R_lock=(_r1, "mean"), n_lock=(_r1, "size"),
                         **{c: (c, "first") for c in _porta}))
    _fcol = _col(CONTR, "f", "f_arm", "freq")
    if _fcol is None:
        raise KeyError("Forma lunga senza colonna di frequenza del braccio: i due controlli "
                       "non si possono ordinare in basso e alto. Colonne: "
                       + ", ".join(CONTR.columns))
    _C = CONTR[~_lk].sort_values(_chiave + [_fcol])
    _n = _C.groupby(_chiave, as_index=False).size().rename(columns={"size": "n_ctrl"})
    # head/tail e non nth: stabile fra le versioni di pandas. Ordinati per frequenza, il primo
    # e' il controllo sotto la riga e l'ultimo quello sopra.
    _clo = _C.groupby(_chiave, sort=False).head(1)[_chiave + [_r1]].rename(columns={_r1: "R_lo"})
    _chi = _C.groupby(_chiave, sort=False).tail(1)[_chiave + [_r1]].rename(columns={_r1: "R_hi"})
    _T = _A.merge(_clo, on=_chiave).merge(_chi, on=_chiave).merge(_n, on=_chiave)
    _bad = _T[(_T.n_lock != 1) | (_T.n_ctrl != 2)]
    if len(_bad):
        print(f"   {len(_bad):,} gruppi su {len(_T):,} non sono triplette "
              f"(un lock e due controlli): scartati")
        print("     distribuzione: " + ", ".join(
            f"{a} lock/{b} ctrl: {c}" for (a, b), c in
            _bad.groupby(['n_lock', 'n_ctrl']).size().items()))
    _T = _T[(_T.n_lock == 1) & (_T.n_ctrl == 2)].drop(columns=["n_lock", "n_ctrl"])
    _T["d"] = _log1(_T.R_lock) - 0.5*(_log1(_T.R_lo) + _log1(_T.R_hi))
    _T["live"] = np.isfinite(_T.d)
    print(f"   triplette ricostruite: {len(_T):,} da {len(CONTR):,} bracci "
          f"(attese circa {len(CONTR)//3:,})")
    CONTR = _T.reset_index(drop=True)

elif FORMA == "larga":
    _dc = _log1(CONTR[_rl]) - 0.5*(_log1(CONTR[_lo]) + _log1(CONTR[_hi]))
    if "d" in CONTR.columns:
        _ok = np.isfinite(CONTR.d.to_numpy(float)) & np.isfinite(_dc)
        _sc = float(np.nanmax(np.abs(CONTR.d.to_numpy(float)[_ok] - _dc[_ok]))) if _ok.any() else np.nan
        print(f"   'd' presente e ricalcolabile: scarto massimo {_sc:.2e}"
              + ("   (coincidono)" if np.isfinite(_sc) and _sc < 1e-9 else "   <- NON coincidono"))
    else:
        CONTR["d"] = _dc; print("   'd' assente: ricalcolata dall'Eq. (d2contrast)")
    if "live" not in CONTR.columns:
        CONTR["live"] = np.isfinite(CONTR.d.to_numpy(float))
        print("   'live' assente: derivata dalla finitezza di d")

else:
    raise KeyError(
        "Forma dei contrasti non riconosciuta. Attesa o la forma larga (R_lock con R_lo e R_hi)\n"
        "   o quella lunga (una colonna R piu' is_lock o role).\n"
        "   Colonne presenti: " + ", ".join(CONTR.columns))

if "live" not in CONTR.columns:
    CONTR["live"] = np.isfinite(CONTR.d.to_numpy(float))
print(f"   triplette utilizzabili: {int(CONTR.live.sum()):,} su {len(CONTR):,}")

_manca = {
    "contrasts": [c for c in ("model", "P", "S", "overlap", "d", "f_lock", "bg_id", "phase_idx")
                  if c not in CONTR.columns],
    "mdl_cells": [c for c in ("model", "stage", "is_locked", "L_bits") if c not in MDL.columns],
    "collapse":  [c for c in ("model", "P", "S", "f", "z") if c not in COLL.columns],
    "sites":     [c for c in ("model", "branch", "predicted_spacing", "f1", "n_sites")
                  if c not in SITI.columns],
}
_TAB = {"contrasts": CONTR, "mdl_cells": MDL, "collapse": COLL, "sites": SITI}
_manca = {k: v for k, v in _manca.items() if v}
if _manca:
    for k, v in _manca.items():
        print(f"\nMANCANO in {k}: {', '.join(v)}")
        print(f"   colonne presenti: {', '.join(_TAB[k].columns)}")
    raise KeyError("Schema incompleto: i modelli non possono essere costruiti. Vedi sopra.")
print("schema completo per tutti e cinque i modelli.")

# ------------------------------------------------------------------ covariate di configurazione
# Il Deliverable definisce il centraggio: O tilde = (O - Obar)/0.5, logP tilde = logP - mean(logP).
GC = (CONTR.groupby("model", as_index=False)
            .agg(P=("P", "first"), S=("S", "first"), overlap=("overlap", "first"))
            .sort_values("model").reset_index(drop=True))
GC["x_o"] = (GC.overlap - GC.overlap.mean()) / 0.5
GC["x_p"] = np.log(GC.P) - np.log(GC.P).mean()
print(f"\ncovariate di configurazione, centrate sul disegno:")
mostra(GC.round(4))
_r = float(np.corrcoef(GC.x_o, GC.x_p)[0, 1])
print(f"correlazione fra le due leve: {_r:+.3f}   VIF {1/(1-_r**2):.3f}"
      "   (si stimano separatamente)")

## 3, Il motore di campionamento

Un fit lungo che muore a meta' non lascia niente, ed e' successo. Qui ogni fit e' spezzato in
blocchi di catene e ogni blocco finisce su disco appena e' pronto: se la sessione salta si riparte
dal blocco dopo. Le catene sono indipendenti per costruzione, quindi campionarle in gruppi e
concatenarle da' **esattamente** la stessa posteriore di un'unica chiamata, e R-hat continua a
confrontare catene davvero indipendenti.

In [ ]:
#@title 3. Il motore di campionamento con checkpoint  { display-mode: "form" }
# Un fit lungo che muore a meta' non lascia niente. Qui ogni fit e' spezzato in BLOCCO catene
# per volta e ogni blocco viene scritto su disco appena finisce: se la sessione salta si
# riparte dal blocco successivo invece che da zero. Le catene sono indipendenti per
# costruzione - semi diversi, nessuno stato condiviso - quindi campionarle in gruppi separati
# e poi concatenarle da' esattamente la stessa posteriore di un'unica chiamata, e R-hat
# continua a confrontare catene davvero indipendenti.

MOTORE        = "auto"   #@param ["auto", "pymc", "numpyro"]
PARAM         = "auto"   #@param ["auto", "centrata", "non centrata"]
MAX_TREEDEPTH = 12       #@param {type:"integer"}
METRICA_DENSA = False    #@param {type:"boolean"}
PROGRESSO     = True     #@param {type:"boolean"}

# PROGRESSO acceso di proposito. Un fit su 136 mila osservazioni dura ore, e senza barra la cella
# non stampa NIENTE fino alla fine: indistinguibile da un kernel bloccato. La barra costa una
# frazione di secondo e toglie l'unica domanda a cui altrimenti non si puo' rispondere.

# La parametrizzazione si MISURA, non si decide a tavolino, e vale la pena dire perche'.
# La regola comune e' che la forma non centrata serve quando ogni gruppo ha pochi dati - l'imbuto
# di Neal - e che la centrata conviene quando ogni gruppo e' ben determinato. Qui ci sono circa
# 136 mila triplette su 15 configurazioni, 47 armoniche e 100 sfondi: migliaia di osservazioni per
# gruppo, quindi la regola direbbe centrata. Su questi dati la regola SBAGLIA, e di parecchio: nel
# pilota la forma non centrata rende circa tre volte piu' ESS al secondo. Il motivo e' che qui la
# geometria difficile non e' l'imbuto ma la cresta additiva fra bbar e la media degli u_k, e su
# quella le due forme non si comportano come la regola prevede.
# Le due forme sono la STESSA distribuzione a posteriori: si sceglie come percorrerla, non cosa.
# Con PARAM = "auto" il pilota decide e il confronto finisce in 03_parametrizzazione.csv.

def _motore_scelto():
    if MOTORE == "pymc": return None
    try:
        import numpyro, jax                                            # noqa: F401
        return "numpyro"
    except Exception:
        if MOTORE == "numpyro":
            print("  numpyro richiesto ma non importabile: si usa PyMC")
        return None

NUTS_SAMPLER = _motore_scelto()
print(f"motore NUTS: {NUTS_SAMPLER or 'pymc'}"
      + ("   (stesso algoritmo, stessa posteriore, solo piu' veloce)" if NUTS_SAMPLER else ""))

MANIFESTO = dict(
    popolazione=sorted(f"p{P}-s{S}" for P, S in POPOLAZIONE),
    banda=list(BANDA), draws=int(DRAWS), tune=int(TUNE), chains=int(CHAINS),
    seed=int(SEED), nu=int(NU), rapido=bool(MODO_RAPIDO),
    righe=dict(contrasts=int(len(CONTR)), mdl=int(len(MDL)),
               collapse=int(len(COLL)), sites=int(len(SITI))),
)
_mp = Path(OUT_DIR) / "manifesto.json"
RIUSA = True
if _mp.exists():
    _v = json.loads(_mp.read_text())
    _diff = {k: (_v.get(k), x) for k, x in MANIFESTO.items() if _v.get(k) != x}
    if _diff:
        print("\nMANIFESTO DIVERSO dai checkpoint presenti: non sono riusabili.")
        for k, (a, b) in _diff.items(): print(f"   {k}: in cache {a!r}, ora {b!r}")
        print("   -> i fit verranno rifatti e i checkpoint sovrascritti")
        RIUSA = False
        _mp.write_text(json.dumps(MANIFESTO, indent=2))
    else:
        print("\nmanifesto identico: i checkpoint gia' presenti verranno riusati")
else:
    _mp.write_text(json.dumps(MANIFESTO, indent=2))

NON_CENTRATO = (PARAM != "centrata")   # provvisorio: con PARAM="auto" lo decide il pilota

FIT = {}        # registro dei fit: la Parte 5 gira su questo, non sui globals
GATE_DETT = []


def _scrivi_checkpoint(idt, f, chiave):
    """Salva un blocco di catene. Non solleva MAI: un fit costato ore non si butta via perche'
    la scrittura fallisce.

    NetCDF passa da h5netcdf, che a sua volta ha bisogno di h5py: se manca, l'ImportError arriva
    a campionamento gia' fatto. In quel caso si ripiega su pickle -- meno portabile, ma contiene
    gli stessi draw -- e se anche quello fallisce si prosegue in memoria, dicendolo chiaramente:
    il notebook arriva in fondo lo stesso, solo senza rete di sicurezza.
    """
    try:
        idt.to_netcdf(f); return idt
    except Exception as e:
        print(f"   [{chiave}] checkpoint NetCDF non scritto: {type(e).__name__}: {e}")
        if "h5py" in str(e):
            print(f"   [{chiave}] manca h5py, il backend di h5netcdf. La cella 1a lo installa:")
            print(f"   [{chiave}] rieseguirla, riavviare il runtime, e i blocchi ripartiranno.")
    try:
        import pickle
        with open(f.with_suffix(".pkl"), "wb") as fh: pickle.dump(idt, fh, protocol=4)
        print(f"   [{chiave}] salvato di ripiego in {f.with_suffix('.pkl').name}")
    except Exception as e2:
        print(f"   [{chiave}] nemmeno il ripiego pickle ha funzionato ({type(e2).__name__}).")
        print(f"   [{chiave}] Il fit e' in memoria e il notebook prosegue, ma se la sessione")
        print(f"   [{chiave}] cade queste catene si perdono.")
    return idt


def _leggi_checkpoint(f):
    """L'inverso: prima il NetCDF, poi il pickle di ripiego. None se non c'e' niente."""
    if f.exists():
        return az.from_netcdf(f)
    g = f.with_suffix(".pkl")
    if g.exists():
        import pickle
        with open(g, "rb") as fh: return pickle.load(fh)
    return None

def campiona(chiave, costruisci, draws=None, tune=None, target_accept=0.90,
             catene=None, loglik=False, metrica_densa=False, seme=None, silenzioso=False,
             max_treedepth=None):
    """Campiona a blocchi di catene, con ripresa dai blocchi gia' su disco.

    costruisci() deve restituire un pm.Model nuovo a ogni chiamata.
    metrica_densa adatta una matrice di massa piena: serve quando la posteriore ha una cresta
    lineare fra parametri, che e' il caso di A e C, e non cambia la distribuzione campionata.
    """
    draws = DRAWS if draws is None else draws
    tune  = TUNE  if tune  is None else tune
    catene = CHAINS if catene is None else catene
    seme = SEED if seme is None else seme
    dirc = Path(OUT_DIR) / "ckpt"
    parti, rifatti = [], 0
    t0 = time.time()
    for c0 in range(0, catene, BLOCCO):
        n = min(BLOCCO, catene - c0)
        f = dirc / f"{chiave}_c{c0:02d}.nc"
        if RIUSA:
            try:
                _vecchio = _leggi_checkpoint(f)
                if _vecchio is not None:
                    parti.append(_vecchio)
                    if not silenzioso: print(f"   [{chiave}] catene {c0}-{c0+n-1}: da checkpoint")
                    continue
            except Exception as e:
                print(f"   [{chiave}] checkpoint {f.name} illeggibile ({type(e).__name__}), rifaccio")
        kw = dict(draws=draws, tune=tune, chains=n, random_seed=seme + c0,
                  target_accept=target_accept, progressbar=PROGRESSO,
                  idata_kwargs={"log_likelihood": loglik})
        if NUTS_SAMPLER:
            # I DUE MOTORI VOGLIONO LA STESSA COSA CON NOMI E ANNIDAMENTI DIVERSI, e sbagliarli
            # non da' un avviso: da' un TypeError a fit gia' avviato, oppure -- peggio -- il
            # parametro viene ignorato in silenzio e si campiona con l'impostazione di default
            # credendo di averla cambiata.
            #   PyMC:     pm.sample(nuts={"max_treedepth": N, ...})
            #   numpyro:  pm.sample(nuts_sampler_kwargs={"nuts_kwargs": {"max_tree_depth": N}})
            # perche' nuts_sampler_kwargs viene passato a sample_jax_nuts(), la cui firma NON ha
            # max_tree_depth: ha nuts_kwargs, che finisce in numpyro.infer.NUTS(**nuts_kwargs).
            # Un livello di annidamento in meno ed e' esattamente l'errore
            #   TypeError: sample_jax_nuts() got an unexpected keyword argument 'max_tree_depth'
            _nuts = {}
            if max_treedepth:  _nuts["max_tree_depth"] = int(max_treedepth)
            # la metrica densa, su questo motore, e' una voce di nuts_kwargs: senza di questa
            # riga METRICA_DENSA non avrebbe alcun effetto quando gira numpyro, e la cresta
            # additiva di A e C resterebbe impercorribile senza che nulla lo segnali.
            if metrica_densa:  _nuts["dense_mass"] = True
            kw.update(nuts_sampler=NUTS_SAMPLER, chain_method="parallel")
            if _nuts: kw["nuts_sampler_kwargs"] = {"nuts_kwargs": _nuts}
        else:
            if max_treedepth:
                kw["nuts"] = {"max_treedepth": int(max_treedepth)}
            kw.update(cores=min(n, CORES))
            if metrica_densa: kw.update(init="jitter+adapt_full")
        with costruisci():
            try:
                idt = pm.sample(**kw)
            except TypeError as e:
                # Le firme dei backend cambiano fra versioni. Se una voce non e' accettata, si
                # ricampiona senza le opzioni di sintonizzazione invece di perdere il fit: la
                # posteriore e' la stessa, cambia solo quanto bene il campionatore la percorre.
                print(f"   [{chiave}] il motore '{NUTS_SAMPLER or 'pymc'}' ha rifiutato le opzioni"
                      f" di sintonizzazione ({e}).")
                print(f"   [{chiave}] ricampiono senza max_treedepth/metrica densa; la posteriore"
                      " e' la stessa, le diagnostiche potrebbero essere peggiori.")
                for _k in ("nuts", "nuts_sampler_kwargs", "init"): kw.pop(_k, None)
                idt = pm.sample(**kw)
        idt = _scrivi_checkpoint(idt, f, chiave)
        parti.append(idt); rifatti += 1
        if not silenzioso:
            print(f"   [{chiave}] catene {c0}-{c0+n-1}: campionate in {time.time()-t0:.0f}s, "
                  f"salvate in {f.name}")
    idata = parti[0] if len(parti) == 1 else az.concat(*parti, dim="chain", reset_dim=True)
    if not silenzioso and rifatti == 0:
        print(f"   [{chiave}] tutto da checkpoint, nessun campionamento")
    return idata

def gates(idata, nomi=None, mostra_peggiori=True):
    """R-hat peggiore, ESS bulk e tail minimi, divergenze. nomi restringe agli estimandi."""
    sel = dict(var_names=list(nomi)) if nomi else {}
    rh_a = az.rhat(idata, **sel)
    eb_a = az.ess(idata, method="bulk", **sel)
    et_a = az.ess(idata, method="tail", **sel)
    def _agg(da, f):
        v = np.asarray(da.to_array().values, float); v = v[np.isfinite(v)]
        return float(f(v)) if v.size else float("nan")
    rh, eb, et = _agg(rh_a, np.max), _agg(eb_a, np.min), _agg(et_a, np.min)
    dv = int(idata.sample_stats.diverging.sum()) if "diverging" in idata.sample_stats else 0
    if mostra_peggiori and (rh > RHAT_MAX or min(eb, et) < ESS_MIN):
        def _peggio(a, asc):
            r = []
            for n, da in a.data_vars.items():
                v = np.asarray(da.values, float).ravel()
                if v.size == 0 or not np.isfinite(v).any(): continue
                i = int(np.nanargmin(v) if asc else np.nanargmax(v))
                r.append((n, float(v[i]), i))
            return sorted(r, key=lambda t: t[1], reverse=not asc)[:4]
        print("     peggiori per R-hat: " + ", ".join(
            f"{n}[{i}]={v:.4f}" for n, v, i in _peggio(rh_a, False)))
        print("     peggiori per ESS:   " + ", ".join(
            f"{n}[{i}]={v:.0f}" for n, v, i in _peggio(eb_a if eb <= et else et_a, True)))
    return rh, min(eb, et), dv, eb, et

def registra(nome, idata, modello=None, dati=None, oss=None, strati=(), estimandi=()):
    FIT[nome] = dict(idata=idata, modello=modello, dati=dati, oss=oss,
                     strati=tuple(strati), estimandi=tuple(estimandi))
    rh, es, dv, eb, et = gates(idata, nomi=estimandi or None)
    GATE_DETT.append(dict(fit=nome, rhat=rh, ess_bulk=eb, ess_tail=et, div=dv,
                          gate_ok=bool(rh <= RHAT_MAX and es >= ESS_MIN and dv <= DIV_MAX)))
    print(f"   [{nome}] R-hat {rh:.4f}  ESS bulk {eb:.0f} tail {et:.0f}  divergenze {dv}"
          f"   -> gate {'PASSATO' if GATE_DETT[-1]['gate_ok'] else 'FALLITO'}")
    return idata

def verdetto(p_sup, p_ref, rhat, ess, div):
    """Tre esiti piu' uno. NON RIPORTABILE riguarda la sola convergenza ed e' separato da
    INCONCLUSIVA, che e' l'esito legittimo di un fit sano ma poco preciso."""
    if not (rhat <= RHAT_MAX and ess >= ESS_MIN and div <= DIV_MAX): return "NON RIPORTABILE"
    if p_sup >= SOGLIA: return "SUPPORTATA"
    if p_ref >= SOGLIA: return "RIFIUTATA"
    return "INCONCLUSIVA"

RIGHE_VERDETTO = []


def scegli_parametrizzazione(costruisci, n=250, righe=None, righe_vere=None):
    """Corre un pilota corto nelle DUE forme e tiene quella che il campionatore percorre meglio.

    Le due forme sono la stessa distribuzione a posteriori: quello che cambia e' la geometria che
    NUTS attraversa, quindi la scelta e' una scelta di campionamento e non di modello. Il criterio
    e' ESS minimo al secondo, che e' quello che conta davvero: una forma due volte piu' lenta ma
    quattro volte piu' efficiente conviene.
    """
    global NON_CENTRATO
    if PARAM == "centrata":
        NON_CENTRATO = False; print("parametrizzazione: CENTRATA (imposta)"); return None
    if PARAM == "non centrata":
        NON_CENTRATO = True;  print("parametrizzazione: NON CENTRATA (imposta)"); return None
    out = []
    # Il pilota e' DUE FIT CORTI il cui unico scopo e' cronometrare le due geometrie. A 250 draws
    # le diagnostiche sono cattive per costruzione -- ESS di poche unita', R-hat sopra 1.01 -- e
    # PyMC le segnala con un record di livello ERROR sul logger pymc.stats.convergence. Quel
    # messaggio, qui, non significa niente: non si sta riportando questa posteriore, si sta solo
    # misurando quale forma il campionatore percorre piu' in fretta. Lo si silenzia per la durata
    # del pilota e lo si rimette subito dopo, perche' sui fit VERI quelle stesse righe vanno viste.
    import logging
    _conv = logging.getLogger("pymc.stats.convergence")
    _liv = _conv.level
    print(f"\npilota: due fit corti da {n} draws, solo per cronometrare le due forme.")
    print("Le diagnostiche del pilota sono cattive per costruzione e NON si riportano: a 250 draws")
    print("ESS basso e R-hat alto sono attesi. I gate valgono sul fit vero, che viene dopo.")
    _conv.setLevel(logging.CRITICAL)
    try:
        for nc in (False, True):
            t0 = time.time()
            try:
                with costruisci(non_centrato=nc):
                    idt = pm.sample(draws=n, tune=n, chains=2, cores=min(2, CORES),
                                    target_accept=0.9, random_seed=SEED + 99, progressbar=False,
                                    **({"nuts_sampler": NUTS_SAMPLER, "chain_method": "parallel"}
                                       if NUTS_SAMPLER else {}))
                dt = time.time() - t0
                eb = float(az.ess(idt, method="bulk").to_array().min())
                et = float(az.ess(idt, method="tail").to_array().min())
                rh = float(az.rhat(idt).to_array().max())
                dv = int(idt.sample_stats.diverging.sum())
                out.append(dict(forma="centrata" if not nc else "non centrata", secondi=round(dt, 1),
                                ess_min=round(min(eb, et), 1), rhat=round(rh, 4), div=dv,
                                ess_al_secondo=round(min(eb, et)/max(dt, 1e-9), 2)))
                del idt; gc.collect()
            except Exception as e:
                out.append(dict(forma="centrata" if not nc else "non centrata", secondi=np.nan,
                                ess_min=np.nan, rhat=np.nan, div=-1, ess_al_secondo=-1.0,
                                errore=f"{type(e).__name__}"))
    finally:
        _conv.setLevel(_liv)      # i fit veri devono tornare a segnalare
    T = pd.DataFrame(out)
    print("\npilota sulle due parametrizzazioni (stessa posteriore, geometrie diverse):")
    mostra(T)
    _best = T.loc[T.ess_al_secondo.idxmax()]
    NON_CENTRATO = bool(_best.forma == "non centrata")
    print(f"-> si usa la forma {_best.forma.upper()}: {_best.ess_al_secondo} ESS al secondo "
          f"contro {T.ess_al_secondo.min()}")
    print("   (numeri del pilota, non del fit: servivano solo a scegliere la forma)")
    if righe and righe_vere:
        stima_dal_pilota(float(_best.secondi), n, righe, righe_vere)
    T.to_csv(f"{OUT_DIR}/03_parametrizzazione.csv", index=False)
    return T


def stima_dal_pilota(secondi_pilota, n_pilota, righe_pilota, righe_vere,
                     draws=None, tune=None, catene=None):
    """Quanto durera' il fit vero, a partire da quanto e' durato il pilota.

    Il costo di NUTS e' circa lineare nel numero di iterazioni e nel numero di osservazioni: il
    gradiente si somma riga per riga. La stima e' grossolana -- non tiene conto del fatto che a
    geometria diversa corrisponde un numero diverso di passi per iterazione -- ma distingue le
    due cose che contano davvero: dieci minuti da dieci ore. Senza, una cella che tace per ore
    e' indistinguibile da una bloccata, ed e' l'unica domanda che poi si finisce per fare.
    """
    draws  = DRAWS  if draws  is None else draws
    tune   = TUNE   if tune   is None else tune
    catene = CHAINS if catene is None else catene
    if not (righe_vere and righe_pilota and secondi_pilota > 0):
        return None
    per_iterazione = secondi_pilota / (2 * n_pilota * 2)        # 2 catene x (tune + draws = 2n)
    sec = per_iterazione * (draws + tune) * catene * (righe_vere / righe_pilota)
    ore, resto = divmod(sec, 3600)
    print(f"\nstima grossolana del fit vero: {catene} catene x {draws}+{tune} iterazioni su "
          f"{righe_vere:,} righe")
    print(f"   ~ {int(ore)}h {int(resto//60):02d}m in totale, a blocchi di {BLOCCO} catene "
          f"(~{sec/max(catene//max(BLOCCO,1),1)/3600:.1f}h per blocco, poi un checkpoint su disco).")
    if sec > 3*3600:
        print("   OLTRE TRE ORE: su Colab conviene un runtime con GPU (numpyro gira in CUDA e")
        print("   cambia di ordine di grandezza), oppure abbassare DRAWS/TUNE nella cella 1b.")
        print("   Se la sessione cade, i blocchi gia' scritti non si rifanno.")
        print("   E questo e' SOLO il modello A: C, B, D1, D2 e la scala dei priori vengono dopo.")
    return sec

## 4, I priori, e la verifica della colonna *Prior*

`tab:bayesDecisions` pubblica, accanto a ogni regola, la probabilita' che quella regola gia'
scorava sotto i priori. E' la cosa che rende leggibile tutto il resto: una posteriore vale come
evidenza solo per quanto si e' mossa da quel numero. Qui la colonna si **ricalcola** dai priori
come sono implementati nel codice, e se non torna il notebook si ferma.

In [ ]:
#@title 4. I priori del Deliverable, e la colonna "Prior" ricalcolata  { display-mode: "form" }
# tab:bayesDecisions dichiara, accanto a ogni regola, la probabilita' che quella regola gia'
# scorava SOTTO I PRIORI. E' la cosa che rende il resto leggibile: una posteriore vale come
# evidenza solo per quanto si e' mossa da quel numero. Qui la colonna si ricalcola per Monte
# Carlo dai priori come sono implementati nel notebook, e si confronta con quella pubblicata.
# Se non tornano, i priori nel codice non sono quelli del testo, ed e' meglio saperlo adesso.

N_PRIOR = 400_000
_rng = np.random.default_rng(SEED)

def _student_t(n, nu=NU, sigma=0.5):
    """Student-t a nu gradi, scala sigma: la parametrizzazione di pm.StudentT."""
    return sigma * _rng.standard_t(nu, size=n)

_bbar   = _student_t(N_PRIOR)                       # A, C: Student-t_4(0, 0.5)
_delta  = _student_t(N_PRIOR)                       # A: delta_O, delta_P, stesso priore
_sphi   = np.abs(_rng.normal(0, 0.25, N_PRIOR))     # C: Half-Normal(0, 0.25)
_thl    = _rng.normal(0, 0.5, N_PRIOR)              # B: Normal(0, 0.5)
_kappa  = _rng.normal(0, 1, N_PRIOR)                # D2: Normal(0, 1)

DICHIARATO = {
    "H1 supportata":  (0.34, float((_bbar < LOG_08).mean())),
    "H1 rifiutata":   (0.14, float((np.abs(_bbar) < LOG_11).mean())),
    "H1 repr. supportata": (0.36, float((_thl > LOG_12).mean())),
    "H1 repr. rifiutata":  (0.15, float((np.abs(_thl) < LOG_11).mean())),
    "H2 supportata":  (0.30, float((_sphi < LOG_11).mean())),
    "H3a, H3b":       (0.05, float((np.abs(_kappa - 1) < ROPE_KAPPA).mean())),
    "M1 supportata":  (0.50, float((_delta > 0).mean())),
}
PRIORI = pd.DataFrame([{"regola": k, "dichiarato": a, "ricalcolato": round(b, 4),
                        "scarto": round(abs(a - b), 4),
                        "esito": "ok" if abs(a - b) <= 0.01 else "NON TORNA"}
                       for k, (a, b) in DICHIARATO.items()])
banner("4. La colonna Prior di tab:bayesDecisions, ricalcolata dai priori implementati")
mostra(PRIORI)
_ko = PRIORI[PRIORI.esito != "ok"]
if len(_ko):
    raise AssertionError(
        "I priori implementati non riproducono la colonna Prior pubblicata: "
        + ", ".join(_ko.regola) + ". Vanno allineati prima di fittare qualunque cosa.")
print("\nI priori nel codice sono quelli del testo. Ogni posteriore piu' avanti va letta come")
print("spostamento da questi numeri, non in assoluto.")
print("\nNota sul segno di M1: la tabella del Deliverable 2 scriveva Pr(delta_O < 0) e la")
print("leggeva come 'overlap reduces the deficit', il che e' contraddittorio su d. La tabella")
print("CONSEGNATA la corregge gia' in Pr(delta_O > 0): qui si segue quella, senza erratum.")

# --------------------------------------------------------------------- predittivo a priori
# Il Deliverable dichiara "Prior predictive checks, before collection". Qui si guarda se i
# priori generano contrasti dell'ordine di grandezza di quelli osservati: un priore che
# produce d di ampiezza 50 non e' debole, e' assurdo, e lo si vede solo guardando.
def _predittivo_a_priori(n=20000):
    bbar = _student_t(n); tau = np.abs(_student_t(n)); sk = np.abs(_student_t(n))
    sb = np.abs(_student_t(n)); sig = np.abs(_student_t(n))
    beta = bbar + tau*_rng.normal(0, 1, n)
    mu   = beta + sk*_rng.normal(0, 1, n) + sb*_rng.normal(0, 1, n)
    return mu + sig*_rng.standard_t(NU, size=n)

_sim = _predittivo_a_priori()
_oss = CONTR.loc[CONTR.live, "d"].to_numpy() if "live" in CONTR.columns else CONTR.d.to_numpy()
print(f"\npredittivo a priori di A, contro i contrasti osservati:")
for _q in (1, 5, 25, 50, 75, 95, 99):
    print(f"   percentile {_q:2d}:  priore {np.percentile(_sim, _q):+8.3f}"
          f"     osservato {np.percentile(_oss, _q):+8.3f}")
print(f"   |d| > 5:      priore {100*np.mean(np.abs(_sim) > 5):5.1f}%"
      f"     osservato {100*np.mean(np.abs(_oss) > 5):5.1f}%")
print("Il priore e' piu' largo dei dati, come deve essere, e non di ordini di grandezza.")

## 5, Modello A, e la lettura M1

$$d_i \sim t_4(\mu_i, \sigma), \qquad \mu_i = \beta_{c[i]} + u_{k[i]} + u_{b[i]}, \qquad
\beta_c \sim \mathcal N\!\big(\bar\beta + \delta_O \widetilde O_c + \delta_P
\widetilde{\log P}_c,\ \tau\big)$$

con $\bar\beta, \delta_O, \delta_P \sim t_4(0, 0.5)$ e
$\tau, \sigma_k, \sigma_b, \sigma \sim \mathrm{Half}\text{-}t_4(0, 0.5)$. Estimando
$e^{\bar\beta}$, previsto $< 1$.

In [ ]:
#@title 5. Modello A, Eq. (d2hier), e la lettura M1  { display-mode: "form" }
# d_i ~ Student-t_4(mu_i, sigma)
# mu_i = beta_{c[i]} + u_{k[i]} + u_{b[i]}
# beta_c ~ Normal(bbar + dO*xo_c + dP*xp_c, tau)
# priori: bbar, dO, dP ~ Student-t_4(0, 0.5);  tau, sk, sb, sigma ~ Half-Student-t_4(0, 0.5)
#
# Scritto non centrato: beta_c = bbar + dO*xo + dP*xp + tau*z_c con z_c ~ N(0,1) e' la STESSA
# distribuzione, non un altro modello. Lo stesso per u_k e u_b.

A_dat = CONTR[CONTR.live].copy() if "live" in CONTR.columns else CONTR.copy()
if MODO_RAPIDO:
    A_dat = A_dat.sample(n=min(12000, len(A_dat)), random_state=SEED).reset_index(drop=True)

_gi = pd.Categorical(A_dat.model, categories=list(GC.model))
A_ci = _gi.codes.astype("int32")
A_ki = pd.Categorical(A_dat.f_lock).codes.astype("int32")
A_bi = pd.Categorical(A_dat.bg_id).codes.astype("int32")
A_y  = A_dat.d.to_numpy(float)
NC, NK, NB = len(GC), int(A_ki.max())+1, int(A_bi.max())+1
XO, XP = GC.x_o.to_numpy(), GC.x_p.to_numpy()

banner("5. Modello A - H1 comportamentale, Eq. (d2hier)")
print(f"triplette: {len(A_y):,}   configurazioni {NC}   armoniche di lock {NK}   sfondi {NB}")
print(f"d osservato: mediana {np.median(A_y):+.4f}   media {A_y.mean():+.4f}   "
      f"sd {A_y.std():.4f}   range [{A_y.min():+.3f}, {A_y.max():+.3f}]")

def modello_A(y=A_y, ci=A_ci, ki=A_ki, bi=A_bi, xo=XO, xp=XP, scala=0.5, con_overlap=True,
              non_centrato=None):
    with pm.Model() as m:
        bbar = pm.StudentT("bbar", nu=NU, mu=0.0, sigma=scala)
        dO   = pm.StudentT("delta_O", nu=NU, mu=0.0, sigma=scala) if con_overlap else 0.0
        dP   = pm.StudentT("delta_P", nu=NU, mu=0.0, sigma=scala)
        tau  = pm.HalfStudentT("tau",     nu=NU, sigma=scala)
        sk   = pm.HalfStudentT("sigma_k", nu=NU, sigma=scala)
        sb   = pm.HalfStudentT("sigma_b", nu=NU, sigma=scala)
        sig  = pm.HalfStudentT("sigma",   nu=NU, sigma=scala)
        nc = NON_CENTRATO if non_centrato is None else non_centrato
        media = bbar + dO*xo + dP*xp
        if nc:
            beta = pm.Deterministic("beta_c", media + tau*pm.Normal("z_c", 0, 1, shape=len(xo)))
            uk = sk*pm.Normal("z_k", 0, 1, shape=int(ki.max())+1)
            ub = sb*pm.Normal("z_b", 0, 1, shape=int(bi.max())+1)
        else:
            beta = pm.Normal("beta_c", mu=media, sigma=tau, shape=len(xo))
            uk = pm.Normal("u_k", 0.0, sk, shape=int(ki.max())+1)
            ub = pm.Normal("u_b", 0.0, sb, shape=int(bi.max())+1)
        # le medie dei due termini incrociati: e' lungo queste che corre la cresta additiva,
        # e averle come quantita' derivate permette di MISURARLA invece di dichiararla
        pm.Deterministic("uk_bar", uk.mean()); pm.Deterministic("ub_bar", ub.mean())
        pm.StudentT("obs", nu=NU, mu=beta[ci] + uk[ki] + ub[bi], sigma=sig, observed=y)
    return m

# quale delle due forme percorrere: pilota corto su un sottocampione, poi il fit vero
_np = min(8000, len(A_y)); _pi_ = np.random.default_rng(SEED+99).choice(len(A_y), _np, replace=False)
PARAM_TAB = scegli_parametrizzazione(
    lambda non_centrato: modello_A(y=A_y[_pi_], ci=A_ci[_pi_], ki=A_ki[_pi_], bi=A_bi[_pi_],
                                   non_centrato=non_centrato),
    righe=_np, righe_vere=len(A_y))

t0 = time.time()
idA = campiona("A", modello_A, target_accept=0.95, metrica_densa=METRICA_DENSA,
               max_treedepth=MAX_TREEDEPTH, loglik=False)
print(f"   fit in {time.time()-t0:.0f}s")
registra("A", idA, modello=modello_A, dati=A_dat, oss=A_y,
         strati=("model", "f_lock", "phase_idx", "generator"),
         estimandi=("bbar", "delta_O", "delta_P", "tau", "sigma_k", "sigma_b", "sigma"))

_b = idA.posterior["bbar"].values.ravel()
_rh, _es, _dv, _, _ = gates(idA, nomi=("bbar",), mostra_peggiori=False)
p_sup = float((_b < LOG_08).mean())            # H1 supportata: almeno 20% di attenuazione
p_ref = float((np.abs(_b) < LOG_11).mean())    # H1 rifiutata: qualunque effetto sotto il 10%
A_riga = dict(modello="A", ipotesi="H1 comportamentale", estimando="e^bbar",
              stima=float(np.exp(_b).mean()),
              lo=float(np.exp(np.percentile(_b, 2.5))), hi=float(np.exp(np.percentile(_b, 97.5))),
              p_sup=p_sup, p_ref=p_ref, rhat=_rh, ess=_es, div=_dv,
              esito=verdetto(p_sup, p_ref, _rh, _es, _dv))
RIGHE_VERDETTO.append(A_riga)
print(f"\n   e^bbar = {A_riga['stima']:.4f}  [{A_riga['lo']:.4f}, {A_riga['hi']:.4f}]")
print(f"   Pr(bbar < log 0.8) = {p_sup:.3f}  (a priori 0.34)   "
      f"Pr(|bbar| < log 1.1) = {p_ref:.3f}  (a priori 0.14)")
print(f"   -> H1 comportamentale {A_riga['esito']}")

# ------------------------------------------------------------------ M1, la lettura di A'
# M1 non e' un modello: e' delta_O di A promosso a estimando. La regola pubblicata ha due
# gambe, la probabilita' e un confronto LOO, e qui si riportano entrambe.
_dO = idA.posterior["delta_O"].values.ravel()
_rhm, _esm, _dvm, _, _ = gates(idA, nomi=("delta_O",), mostra_peggiori=False)
m_sup = float((_dO > 0).mean())      # ERRATUM: il testo scrive < 0, la sua glossa implica > 0
m_ref = float((_dO < 0).mean())
M1_riga = dict(modello="A' (M1)", ipotesi="M1 mitigazione", estimando="delta_O",
               stima=float(_dO.mean()), lo=float(np.percentile(_dO, 2.5)),
               hi=float(np.percentile(_dO, 97.5)), p_sup=m_sup, p_ref=m_ref,
               rhat=_rhm, ess=_esm, div=_dvm, esito=verdetto(m_sup, m_ref, _rhm, _esm, _dvm))
print(f"\n   delta_O = {M1_riga['stima']:+.4f}  [{M1_riga['lo']:+.4f}, {M1_riga['hi']:+.4f}]")
print(f"   Pr(delta_O > 0) = {m_sup:.3f}  (a priori 0.50)")
print(f"   delta_P = {idA.posterior['delta_P'].values.mean():+.4f}"
      f"  [{np.percentile(idA.posterior['delta_P'].values, 2.5):+.4f},"
      f" {np.percentile(idA.posterior['delta_P'].values, 97.5):+.4f}]")
print("   La seconda gamba della regola - il termine di overlap vince il LOO - e' nella 5.5.")


# ------------------------------------------------------------------ la cresta, misurata
# bbar puo' salire di epsilon se le medie di u_k e u_b scendono di epsilon: mu non si muove.
# Se quella direzione e' quasi piatta, la correlazione a posteriori fra bbar e le due medie e'
# vicina a -1, ed e' il numero che spiega un ESS basso senza divergenze.
_post = idA.posterior
_bb = _post["bbar"].values.ravel()
_uk = _post["uk_bar"].values.ravel(); _ub = _post["ub_bar"].values.ravel()
_rk = float(np.corrcoef(_bb, _uk)[0, 1]); _rb = float(np.corrcoef(_bb, _ub)[0, 1])
_rs = float(np.corrcoef(_bb, _uk + _ub)[0, 1])
print(f"\n   cresta additiva, correlazione a posteriori:")
print(f"     bbar contro media(u_k)          {_rk:+.3f}")
print(f"     bbar contro media(u_b)          {_rb:+.3f}")
print(f"     bbar contro media(u_k)+media(u_b) {_rs:+.3f}")
print(f"     sd di bbar {_bb.std():.4f}   sd della somma bbar+medie {(_bb+_uk+_ub).std():.4f}")
if _rs < -0.8:
    print("     La direzione additiva e' quasi piatta: e' una proprieta' di Eq. (d2hier), non")
    print("     del campionatore, e la larghezza di bbar che ne deriva E' il risultato.")
CRESTA = dict(r_uk=_rk, r_ub=_rb, r_somma=_rs, sd_bbar=float(_bb.std()),
              sd_somma=float((_bb+_uk+_ub).std()))
pd.DataFrame([CRESTA]).to_csv(f"{OUT_DIR}/05_cresta.csv", index=False)

## 6, Modello C

$$y_i \sim t_4(\mu_i, \sigma), \qquad \mu_i = \beta_{c[i]} + u_{k[i]} + u_{b[i]} + u_{p[i]},
\qquad u_p \sim \mathcal N(0, \sigma_\phi)$$

su $y = -d$, senza le covariate di configurazione, con $\sigma_\phi \sim \mathrm{Half}\text{-}
\mathcal N(0, 0.25)$. Estimando $\sigma_\phi$, previsto $\approx 0$.

In [ ]:
#@title 6. Modello C, Eq. (d2phase) - H2  { display-mode: "form" }
# y_i ~ Student-t_4(mu_i, sigma),  mu_i = beta_{c[i]} + u_{k[i]} + u_{b[i]} + u_{p[i]}
# u_p ~ Normal(0, sigma_phi),  sigma_phi ~ Half-Normal(0, 0.25)
# E' il Modello A su y = -d, SENZA le covariate di configurazione e con un offset per fetta di fase.
#
# ERRATUM: il testo dice otto fette, i dati raccolti ne hanno dieci. Il modello resta "un offset
# per fetta"; cambia il numero di fette. Con dieci gruppi invece di otto sigma_phi si stima un
# po' meglio, non peggio.

C_y  = -A_dat.d.to_numpy(float)            # y = d col segno rovesciato, come dice il Deliverable
C_pi = pd.Categorical(A_dat.phase_idx).codes.astype("int32")
NPH  = int(C_pi.max()) + 1
banner("6. Modello C - H2, invarianza di fase, Eq. (d2phase)")
print(f"fette di fase nei dati: {NPH}   (il testo del Deliverable dice 8: erratum dichiarato)")
assert np.allclose(C_y, -A_dat.d.to_numpy(float))

def modello_C(y=C_y, ci=A_ci, ki=A_ki, bi=A_bi, pi=C_pi, scala=0.5, scala_phi=0.25,
              con_fase=True, non_centrato=None):
    with pm.Model() as m:
        bbar = pm.StudentT("bbar", nu=NU, mu=0.0, sigma=scala)
        tau  = pm.HalfStudentT("tau",     nu=NU, sigma=scala)
        sk   = pm.HalfStudentT("sigma_k", nu=NU, sigma=scala)
        sb   = pm.HalfStudentT("sigma_b", nu=NU, sigma=scala)
        sig  = pm.HalfStudentT("sigma",   nu=NU, sigma=scala)
        nc = NON_CENTRATO if non_centrato is None else non_centrato
        if nc:
            beta = bbar + tau*pm.Normal("z_c", 0, 1, shape=int(ci.max())+1)
            uk = sk*pm.Normal("z_k", 0, 1, shape=int(ki.max())+1)
            ub = sb*pm.Normal("z_b", 0, 1, shape=int(bi.max())+1)
        else:
            beta = pm.Normal("beta_c", mu=bbar, sigma=tau, shape=int(ci.max())+1)
            uk = pm.Normal("u_k", 0.0, sk, shape=int(ki.max())+1)
            ub = pm.Normal("u_b", 0.0, sb, shape=int(bi.max())+1)
        mu = beta[ci] + uk[ki] + ub[bi]
        if con_fase:
            sphi = pm.HalfNormal("sigma_phi", sigma=scala_phi)
            up = (sphi*pm.Normal("z_p", 0, 1, shape=int(pi.max())+1) if nc
                  else pm.Normal("u_p_raw", 0.0, sphi, shape=int(pi.max())+1))
            pm.Deterministic("u_p", up)
            mu = mu + up[pi]
        pm.StudentT("obs", nu=NU, mu=mu, sigma=sig, observed=y)
    return m

t0 = time.time()
idC = campiona("C", modello_C, target_accept=0.95, metrica_densa=METRICA_DENSA,
               max_treedepth=MAX_TREEDEPTH)
print(f"   fit in {time.time()-t0:.0f}s")
registra("C", idC, modello=modello_C, dati=A_dat, oss=C_y,
         strati=("model", "f_lock", "phase_idx", "generator"),
         estimandi=("bbar", "sigma_phi", "tau", "sigma_k", "sigma_b", "sigma"))

_sp = idC.posterior["sigma_phi"].values.ravel()
_rh, _es, _dv, _, _ = gates(idC, nomi=("sigma_phi",), mostra_peggiori=False)
c_sup = float((_sp < LOG_11).mean())     # H2 supportata: la fase muove il deficit sotto il 10%
c_ref = float((_sp > LOG_11).mean())
C_riga = dict(modello="C", ipotesi="H2", estimando="sigma_phi", stima=float(_sp.mean()),
              lo=float(np.percentile(_sp, 2.5)), hi=float(np.percentile(_sp, 97.5)),
              p_sup=c_sup, p_ref=c_ref, rhat=_rh, ess=_es, div=_dv,
              esito=verdetto(c_sup, c_ref, _rh, _es, _dv))
RIGHE_VERDETTO.append(C_riga)
print(f"\n   sigma_phi = {C_riga['stima']:.4f}  [{C_riga['lo']:.4f}, {C_riga['hi']:.4f}]"
      f"   (soglia log 1.1 = {LOG_11:.4f})")
print(f"   Pr(sigma_phi < log 1.1) = {c_sup:.3f}  (a priori 0.30)")
print(f"   -> H2 {C_riga['esito']}")
_up = idC.posterior["u_p"].values.reshape(-1, NPH)
print(f"\n   offset per fetta, media a posteriori: "
      + " ".join(f"{v:+.3f}" for v in _up.mean(0)))
print(f"   con {NPH} gruppi la sd a posteriori di log(sigma_phi) non puo' scendere sotto")
print(f"   circa 1/sqrt(2*({NPH}-1)) = {1/np.sqrt(2*(NPH-1)):.3f}: e' un limite del disegno,")
print("   non del campionamento, e vale la pena dirlo accanto al numero.")

## 7, Modello B

$$L_i \sim \mathrm{Gamma}(k,\ k/\mu_i), \qquad
\log \mu_i = \alpha_0 + \theta_{\mathrm{lock}} \mathrm{IsLocked}_i + u_{st[i]} + u_{geo[i]}$$

Estimando $e^{\theta_{\mathrm{lock}}}$, previsto $> 1$. **Il Deliverable non gli assegna una
soglia**, e qui non se ne inventa una.

In [ ]:
#@title 7. Modello B - H1 rappresentazionale  { display-mode: "form" }
# L_i ~ Gamma(k, k/mu_i),  log mu_i = alpha0 + theta_lock*IsLocked + u_st + u_geo
# priori: alpha0 ~ N(log Lbar, 1), theta_lock ~ N(0, 0.5), k ~ Gamma(2, 0.1),
#         sigma_st, sigma_geo ~ Half-Normal(0, 1)
#
# La tabella del Deliverable 2 non dava a B nessuna soglia: le due righe di H1 erano scritte
# entrambe in bbar, che e' il parametro di A. La tabella CONSEGNATA colma quella lacuna e fissa
# due regole per la meta' rappresentazionale, che sono quelle applicate qui:
#     supportata  Pr(theta_lock > log 1.2 | D) >= 0.95     a priori 0.36
#     rifiutata   Pr(|theta_lock| < log 1.1 | D) >= 0.95   a priori 0.15

B_dat = MDL[np.isfinite(MDL.L_bits) & (MDL.L_bits > 0)].copy()
B_L  = B_dat.L_bits.to_numpy(float)
B_lk = B_dat.is_locked.to_numpy(float)
B_si = pd.Categorical(B_dat.stage).codes.astype("int32")
B_gi = pd.Categorical(B_dat.model).codes.astype("int32")
banner("7. Modello B - H1 rappresentazionale")
print(f"celle {len(B_dat):,}   stadi di probe {B_si.max()+1}   geometrie {B_gi.max()+1}")
print(f"codelength: mediana {np.median(B_L):.2f} bit   range [{B_L.min():.2f}, {B_L.max():.2f}]")
print(f"quota di celle a lock: {B_lk.mean():.3f}")

def modello_B(L=B_L, lk=B_lk, si=B_si, gi=B_gi, scala=0.5):
    with pm.Model() as m:
        a0 = pm.Normal("alpha0", np.log(L.mean()), 1.0)
        th = pm.Normal("theta_lock", 0.0, scala)
        k  = pm.Gamma("k", alpha=2.0, beta=0.1)
        ss = pm.HalfNormal("sigma_st", 1.0);  zs = pm.Normal("z_st", 0, 1, shape=int(si.max())+1)
        sg = pm.HalfNormal("sigma_geo", 1.0); zg = pm.Normal("z_geo", 0, 1, shape=int(gi.max())+1)
        mu = pm.math.exp(a0 + th*lk + ss*zs[si] + sg*zg[gi])
        pm.Gamma("obs", alpha=k, beta=k/mu, observed=L)
    return m

t0 = time.time()
idB = campiona("B", modello_B, target_accept=0.95)
print(f"   fit in {time.time()-t0:.0f}s")
registra("B", idB, modello=modello_B, dati=B_dat, oss=B_L,
         strati=("model", "stage"), estimandi=("alpha0", "theta_lock", "k", "sigma_st", "sigma_geo"))

_th = idB.posterior["theta_lock"].values.ravel()
_rh, _es, _dv, _, _ = gates(idB, nomi=("theta_lock",), mostra_peggiori=False)
b_sup = float((_th > LOG_12).mean())            # almeno il 20% di codice in piu'
b_ref = float((np.abs(_th) < LOG_11).mean())   # qualunque espansione sotto il 10%
B_riga = dict(modello="B", ipotesi="H1 rappresentazionale", estimando="e^theta_lock",
              stima=float(np.exp(_th).mean()),
              lo=float(np.exp(np.percentile(_th, 2.5))), hi=float(np.exp(np.percentile(_th, 97.5))),
              p_sup=b_sup, p_ref=b_ref, rhat=_rh, ess=_es, div=_dv,
              esito=verdetto(b_sup, b_ref, _rh, _es, _dv))
RIGHE_VERDETTO.append(B_riga)
print(f"\n   e^theta_lock = {B_riga['stima']:.4f}  [{B_riga['lo']:.4f}, {B_riga['hi']:.4f}]")
print(f"   Pr(theta_lock > log 1.2) = {b_sup:.3f}  (a priori 0.36)   "
      f"Pr(|theta_lock| < log 1.1) = {b_ref:.3f}  (a priori 0.15)")
print(f"   -> H1 rappresentazionale {B_riga['esito']}")

## 8, Modello D1

$$\log z_g(f) \sim \mathcal N\!\big(\alpha_g + \theta_S \cdot \mathrm{OnStrideGrid}
+ \theta_P \cdot \mathrm{OnPatchGrid},\ \sigma\big)$$

Regola **congiunta**: il fit a due etichette vince il LOO *e* $\theta_S, \theta_P < 0$.

In [ ]:
#@title 8. Modello D1, Eq. (d2sites) - H3, dove stanno i siti  { display-mode: "form" }
# log z_g(f) ~ Normal(alpha_g + theta_S*OnStrideGrid + theta_P*OnPatchGrid, sigma)
# priori: alpha_g, theta_S, theta_P ~ Normal(0,1);  sigma ~ Half-Normal(0,1)
#
# La risposta e' log z, come scrive il Deliverable, NON log z_norm. Costo dichiarato: alpha_g e'
# uno per geometria mentre log z varia anche con modo e replica, quindi quella variazione finisce
# nella varianza residua e diluisce theta_S e theta_P. E' la lettera del modello consegnato.

TOL_GRID = 0.5   #@param {type:"number"}   meta' passo dello sweep: ogni sito marca un solo bin

D1_dat = COLL[(COLL.f >= BANDA[0]) & (COLL.f <= BANDA[1])].copy()
_f = D1_dat.f.to_numpy(float); _P = D1_dat.P.to_numpy(float); _S = D1_dat.S.to_numpy(float)
_kS = np.round(_f*_S/FS); _kP = np.round(_f*_P/FS)
onS = ((_kS >= 1) & (np.abs(_f - _kS*FS/_S) <= TOL_GRID)).astype(float)
onP = ((_kP >= 1) & (np.abs(_f - _kP*FS/_P) <= TOL_GRID)).astype(float)
D1_y = np.log(D1_dat.z.to_numpy(float))
D1_gi = pd.Categorical(D1_dat.model).codes.astype("int32")
D1_dat = D1_dat.assign(onS=onS, onP=onP)

banner("8. Modello D1 - H3, Eq. (d2sites)")
print(f"righe {len(D1_dat):,}   geometrie {D1_gi.max()+1}   banda [{BANDA[0]:.0f}, {BANDA[1]:.0f}] Hz")
print(f"log z: mediana {np.median(D1_y):+.3f}   range [{D1_y.min():+.3f}, {D1_y.max():+.3f}]")
print(f"siti marcati con tolleranza {TOL_GRID} Hz:  stride {int(onS.sum()):,}"
      f"   patch {int(onP.sum()):,}   entrambi {int((onS*onP).sum()):,}")
if D1_y.min() < -6:
    print(f"   NOTA: la coda bassa di log z arriva a {D1_y.min():.2f}. La verosimiglianza di")
    print("   Eq. (d2sites) e' Normale, quindi quei punti pesano al quadrato: la 5.2 li guarda.")

def modello_D1(y=D1_y, gi=D1_gi, s=onS, p=onP, scala=1.0, usa_S=True, usa_P=True):
    with pm.Model() as m:
        a  = pm.Normal("alpha_geo", 0.0, scala, shape=int(gi.max())+1)
        tS = pm.Normal("theta_S", 0.0, scala) if usa_S else 0.0
        tP = pm.Normal("theta_P", 0.0, scala) if usa_P else 0.0
        sd = pm.HalfNormal("sigma", 1.0)
        pm.Normal("obs", mu=a[gi] + tS*s + tP*p, sigma=sd, observed=y)
    return m

t0 = time.time()
idD1 = campiona("D1", modello_D1, target_accept=0.90)
print(f"   fit in {time.time()-t0:.0f}s")
registra("D1", idD1, modello=modello_D1, dati=D1_dat, oss=D1_y,
         strati=("model", "mode"), estimandi=("theta_S", "theta_P", "sigma"))

D1_righe = []
for _nm, _lab in (("S", "H3 ramo stride"), ("P", "H3 ramo patch")):
    _t = idD1.posterior[f"theta_{_nm}"].values.ravel()
    _rh, _es, _dv, _, _ = gates(idD1, nomi=(f"theta_{_nm}",), mostra_peggiori=False)
    D1_righe.append(dict(modello=f"D1 ramo {_nm}", ipotesi=_lab, estimando=f"theta_{_nm}",
                         stima=float(_t.mean()), lo=float(np.percentile(_t, 2.5)),
                         hi=float(np.percentile(_t, 97.5)),
                         p_sup=float((_t < 0).mean()), p_ref=float((_t > 0).mean()),
                         rhat=_rh, ess=_es, div=_dv, esito="(nel gate congiunto)"))
    print(f"   theta_{_nm} = {D1_righe[-1]['stima']:+.4f}"
          f"  [{D1_righe[-1]['lo']:+.4f}, {D1_righe[-1]['hi']:+.4f}]"
          f"   Pr(<0) = {D1_righe[-1]['p_sup']:.3f}")
print("\n   La regola del Deliverable e' CONGIUNTA: il fit a due etichette vince il LOO e")
print("   theta_S, theta_P sono entrambi negativi. Il verdetto si forma nella 5.5, dopo il LOO.")

## 9, Modello D2

$$\hat f^{\,F}_{1,g} \sim \mathcal N\!\big(\kappa_F \Delta^F_g,\
\sqrt{\sigma_F^2 + \Delta f^2}\big)$$

Regola: $\Pr(|\kappa_F - 1| < 0.1) \ge 0.95$, e almeno dieci siti non ambigui perche' il ramo
sia considerato identificato.

In [ ]:
#@title 9. Modello D2, Eq. (d2move) - H3a e H3b, se i siti si spostano  { display-mode: "form" }
# f1_g^F ~ Normal(kappa_F * Delta_g^F, sqrt(sigma_F^2 + Delta_f^2))
# priori: kappa_F ~ Normal(0,1);  sigma_F ~ Half-Normal(0,5) Hz
# Il floor Delta_f = passo dello sweep: una spaziatura letta su una griglia discreta non e' piu'
# precisa di un passo, e senza il floor i siti esattamente sulla predizione portano sigma_F a zero.
#
# Condizione di identificazione dichiarata: almeno 10 siti non ambigui per il ramo, altrimenti
# il ramo si riporta come NON IDENTIFICATO invece di essere stimato.

banner("9. Modello D2 - H3a e H3b, Eq. (d2move)")
D2_righe = []
for _br, _ip in (("stride", "H3a (siti ~ 1/S)"), ("patch", "H3b (siti ~ 1/P)")):
    _s = SITI[(SITI.branch == _br) & SITI.f1.notna()]
    _n_siti = int(SITI.loc[SITI.branch == _br, "n_sites"].sum())
    # "il fondamentale di quel ramo in quella geometria": uno per geometria, mediana su modo e replica
    _g = (_s.groupby("model", as_index=False)
            .agg(f1=("f1", "median"), spac=("predicted_spacing", "first"), n=("f1", "size")))
    print(f"\n--- ramo {_br}: {len(_g)} geometrie con un fondamentale, "
          f"{_n_siti} siti non ambigui in totale")
    if len(_g):
        mostra(_g.assign(kappa_emp=(_g.f1/_g.spac).round(4)).round(3))
    if _n_siti < SITI_MIN:
        print(f"   {_n_siti} < {SITI_MIN}: la condizione di identificazione del Deliverable non e'")
        print(f"   soddisfatta. Il ramo si riporta NON IDENTIFICATO e non viene stimato.")
        D2_righe.append(dict(modello=f"D2 ramo {_br}", ipotesi=_ip, estimando=f"kappa_{_br[0].upper()}",
                             stima=np.nan, lo=np.nan, hi=np.nan, p_sup=np.nan, p_ref=np.nan,
                             rhat=np.nan, ess=np.nan, div=0, esito="NON IDENTIFICATO",
                             nota=f"{_n_siti} siti non ambigui, minimo {SITI_MIN}"))
        continue
    if len(_g) < 3:
        print(f"   solo {len(_g)} geometrie: troppe poche per una pendenza. NON IDENTIFICATO.")
        D2_righe.append(dict(modello=f"D2 ramo {_br}", ipotesi=_ip, estimando=f"kappa_{_br[0].upper()}",
                             stima=np.nan, lo=np.nan, hi=np.nan, p_sup=np.nan, p_ref=np.nan,
                             rhat=np.nan, ess=np.nan, div=0, esito="NON IDENTIFICATO",
                             nota=f"{len(_g)} geometrie"))
        continue
    _y = _g.f1.to_numpy(float); _x = _g.spac.to_numpy(float)
    def _mk(y=_y, x=_x, scala=1.0):
        def _f():
            with pm.Model() as m:
                kp = pm.Normal("kappa", 0.0, scala)
                sF = pm.HalfNormal("sigma_F", 5.0)
                pm.Normal("obs", mu=kp*x, sigma=pm.math.sqrt(sF**2 + DELTA_F**2), observed=y)
            return m
        return _f
    idk = campiona(f"D2_{_br}", _mk(), target_accept=0.95)
    registra(f"D2_{_br}", idk, dati=_g, oss=_y, estimandi=("kappa", "sigma_F"))
    _k = idk.posterior["kappa"].values.ravel()
    _rh, _es, _dv, _, _ = gates(idk, nomi=("kappa",), mostra_peggiori=False)
    _sup = float((np.abs(_k - 1) < ROPE_KAPPA).mean())
    _ref = float((np.abs(_k - 1) >= ROPE_KAPPA).mean())
    D2_righe.append(dict(modello=f"D2 ramo {_br}", ipotesi=_ip, estimando=f"kappa_{_br[0].upper()}",
                         stima=float(_k.mean()), lo=float(np.percentile(_k, 2.5)),
                         hi=float(np.percentile(_k, 97.5)), p_sup=_sup, p_ref=_ref,
                         rhat=_rh, ess=_es, div=_dv,
                         esito=verdetto(_sup, _ref, _rh, _es, _dv), nota=f"{_n_siti} siti"))
    print(f"   kappa = {D2_righe[-1]['stima']:.4f}  [{D2_righe[-1]['lo']:.4f},"
          f" {D2_righe[-1]['hi']:.4f}]   Pr(|kappa-1| < 0.1) = {_sup:.3f}  (a priori 0.05)")
    print(f"   -> {_ip} {D2_righe[-1]['esito']}")

RIGHE_VERDETTO.extend(D1_righe); RIGHE_VERDETTO.extend(D2_righe)

# Parte 5, i controlli che il Deliverable ha dichiarato

*Prior predictive checks* (sezione 4), *parameter recovery*, *posterior predictive checks
stratified by frequency, phase, P, S and generator*, *model comparison by LOO*, piu' la scala dei
priori $\{0.25, 0.5, 1.0\}$. Nessun controllo in piu', nessuno in meno.

In [ ]:
#@title 10. Convergenza di ogni fit  { display-mode: "form" }
# Il Deliverable 2 dichiara R-hat < 1.01, ESS > 1000, zero divergenze. Sono le soglie a cui il
# lettore e' stato impegnato, quindi restano quelle. Qui si riportano per ogni fit, e su DUE
# diagnostiche di ESS: bulk e tail. La coda e' quella che cede per prima sulle scale gerarchiche.
banner("10. Convergenza")
DIAG = pd.DataFrame(GATE_DETT)
mostra(DIAG.round(4))
DIAG.to_csv(f"{OUT_DIR}/10_convergenza.csv", index=False)
_ko = DIAG[~DIAG.gate_ok]
print(f"\nfit che passano il gate: {int(DIAG.gate_ok.sum())}/{len(DIAG)}")
if len(_ko):
    print("non passano: " + ", ".join(_ko.fit))
    print("Per A e C il sospetto da verificare e' la cresta additiva descritta in testa al")
    print("notebook: se R-hat e' alto, ESS minuscolo e le divergenze sono ZERO, non e' curvatura")
    print("ma una direzione piatta, e va riportata come tale invece che inseguita con piu' draws.")

In [ ]:
#@title 11. Controlli predittivi a posteriori, stratificati  { display-mode: "form" }
# Il Deliverable dichiara PPC "stratified by frequency, phase, P, S and generator". Sono
# esattamente queste le stratificazioni, niente di piu' e niente di meno.
PPC_DRAWS = 300  #@param {type:"integer"}

def _ppc(nome, strati_extra=()):
    f = FIT.get(nome)
    if f is None or f.get("modello") is None: return []
    dati, oss = f["dati"], f["oss"]
    if dati is None or oss is None: return []
    try:
        with f["modello"]():
            pp = pm.sample_posterior_predictive(
                f["idata"], var_names=["obs"], random_seed=SEED,
                progressbar=False, extend_inferencedata=False)
    except Exception as e:
        print(f"   {nome}: PPC saltato ({type(e).__name__}: {e})"); return []
    rep = pp.posterior_predictive["obs"].values.reshape(-1, len(oss))
    if len(rep) > PPC_DRAWS:
        rep = rep[np.linspace(0, len(rep)-1, PPC_DRAWS).astype(int)]
    righe = []
    for col in f["strati"] + tuple(strati_extra):
        if col not in dati.columns: continue
        v = dati[col].to_numpy()
        for liv in pd.unique(v)[:40]:
            sel = (v == liv)
            if sel.sum() < 5: continue
            o = float(np.mean(oss[sel])); r = rep[:, sel].mean(1)
            lo, hi = np.percentile(r, [2.5, 97.5])
            righe.append(dict(fit=nome, strato=col, livello=str(liv), n=int(sel.sum()),
                              osservato=o, rep_lo=float(lo), rep_hi=float(hi),
                              ppc_ok=bool(lo <= o <= hi)))
    return righe

banner("11. Controlli predittivi a posteriori")
_r = []
for _n in ("A", "C", "B", "D1"):
    _rr = _ppc(_n)
    _r += _rr
    if _rr:
        _q = pd.DataFrame(_rr).groupby("strato").ppc_ok.agg(passati="sum", strati="count")
        print(f"\n{_n}:"); mostra(_q)
PPC = pd.DataFrame(_r)
if len(PPC):
    PPC.to_csv(f"{OUT_DIR}/11_ppc.csv", index=False)
    print(f"\ncomplessivo: {int(PPC.ppc_ok.sum())}/{len(PPC)} strati coperti")
    _bad = PPC[~PPC.ppc_ok]
    if len(_bad):
        print("\nstrati non coperti:"); mostra(_bad.head(25).round(4))

def ppc_ok(*nomi, quota=0.95):
    if not len(PPC): return False
    for n in nomi:
        s = PPC[PPC.fit == n]
        if not len(s) or s.ppc_ok.mean() < quota: return False
    return True

In [ ]:
#@title 12. Recupero di parametri  { display-mode: "form" }
# "data simulated with a known effect, to confirm the model finds it. This is what separates
# 'no effect' from 'a design that cannot see one'." Una famiglia di verosimiglianza per volta,
# dati sintetici, criterio: l'intervallo al 95% contiene il vero. NIENTE QUI E' UN RISULTATO.
REC_DRAWS = max(400, DRAWS // 2)
_rr = np.random.default_rng(SEED + 7)

def _riga_rec(idata, fam, veri):
    out = []
    dv = int(idata.sample_stats.diverging.sum()) if "diverging" in idata.sample_stats else 0
    for p, vero in veri.items():
        if p not in idata.posterior: continue
        d = idata.posterior[p].values.ravel()
        lo, hi = np.percentile(d, [2.5, 97.5])
        rh = float(az.rhat(idata, var_names=[p]).to_array().max())
        eb = float(az.ess(idata, var_names=[p], method="bulk").to_array().min())
        et = float(az.ess(idata, var_names=[p], method="tail").to_array().min())
        out.append(dict(famiglia=fam, parametro=p, vero=float(vero), mediana=float(np.median(d)),
                        lo=float(lo), hi=float(hi), coperto=bool(lo <= vero <= hi),
                        conv_ok=bool(rh <= RHAT_MAX and min(eb, et) >= ESS_MIN and dv <= DIV_MAX),
                        rhat=rh, ess_bulk=eb, ess_tail=et, div=dv))
    return out

banner("12. Recupero di parametri (sintetico, non riportabile)")
REC = []
# --- famiglia Student-t gerarchica: A e C -------------------------------------------------
_n = min(len(A_y), 30000)
_ix = _rr.choice(len(A_y), _n, replace=False)
_ci, _ki, _bi = A_ci[_ix], A_ki[_ix], A_bi[_ix]
_VERI = dict(bbar=-0.30, delta_O=0.20, delta_P=-0.10)
_tau, _sk, _sb, _sg = 0.25, 0.20, 0.15, 0.60
_beta = _VERI["bbar"] + _VERI["delta_O"]*XO + _VERI["delta_P"]*XP + _tau*_rr.normal(0,1,NC)
_mu = (_beta[_ci] + _sk*_rr.normal(0,1,int(_ki.max())+1)[_ki]
       + _sb*_rr.normal(0,1,int(_bi.max())+1)[_bi])
_ysim = _mu + _sg*_rr.standard_t(NU, size=_n)
print(f"Student-t gerarchica: {_n:,} osservazioni simulate dal modello di A")
_id = campiona("rec_A", lambda: modello_A(y=_ysim, ci=_ci, ki=_ki, bi=_bi),
               draws=REC_DRAWS, tune=REC_DRAWS, target_accept=0.95,
               metrica_densa=METRICA_DENSA, max_treedepth=MAX_TREEDEPTH,
               seme=SEED+7, silenzioso=True)
REC += _riga_rec(_id, "Student-t gerarchica", _VERI)
del _id; gc.collect()
# --- famiglia Gamma a link log: B ----------------------------------------------------------
_VB = dict(theta_lock=0.35, alpha0=float(np.log(B_L.mean())))
_muB = np.exp(_VB["alpha0"] + _VB["theta_lock"]*B_lk
              + 0.3*_rr.normal(0,1,int(B_si.max())+1)[B_si]
              + 0.2*_rr.normal(0,1,int(B_gi.max())+1)[B_gi])
_kB = 20.0
_LB = _rr.gamma(shape=_kB, scale=_muB/_kB)
print(f"Gamma a link logaritmico: {len(_LB):,} celle simulate dal modello di B")
_id = campiona("rec_B", lambda: modello_B(L=_LB), draws=REC_DRAWS, tune=REC_DRAWS,
               target_accept=0.95, seme=SEED+7, silenzioso=True)
REC += _riga_rec(_id, "Gamma+log", _VB)
del _id; gc.collect()
# --- famiglia Normale con etichette: D1 ----------------------------------------------------
_VD = dict(theta_S=-0.40, theta_P=-0.25)
_yD = (_rr.normal(0,1,int(D1_gi.max())+1)[D1_gi] + _VD["theta_S"]*onS + _VD["theta_P"]*onP
       + 0.5*_rr.normal(0, 1, len(D1_y)))
print(f"Normale con etichette: {len(_yD):,} righe simulate dal modello di D1")
_id = campiona("rec_D1", lambda: modello_D1(y=_yD), draws=REC_DRAWS, tune=REC_DRAWS,
               target_accept=0.90, seme=SEED+7, silenzioso=True)
REC += _riga_rec(_id, "Normale+etichette", _VD)
del _id; gc.collect()

RECUPERO = pd.DataFrame(REC)
mostra(RECUPERO.round(4))
RECUPERO.to_csv(f"{OUT_DIR}/12_recupero.csv", index=False)
print("\n'coperto' e' la domanda vera: l'intervallo contiene il valore che ha generato i dati.")
print("'conv_ok' riguarda il fit SINTETICO e puo' fallire per il suo tuning senza dire nulla")
print("sul modello riportato: vanno letti separatamente.")

def recupero_ok(fam, pars):
    s = RECUPERO[(RECUPERO.famiglia == fam) & (RECUPERO.parametro.isin(pars))]
    return bool(len(s) and (s.coperto & s.conv_ok).all())

In [ ]:
#@title 13. Confronti LOO  { display-mode: "form" }
# Il Deliverable usa il LOO in tre punti: la regola di H3 ("two-label fit wins LOO"), la seconda
# gamba di M1 ("overlap term wins LOO") e il nullo annidato di C ("dropping u_p gives a nested
# null"). Con 136 mila osservazioni la log-verosimiglianza per punto non sta in memoria, quindi
# i confronti girano su un sottocampione stratificato e a draws ridotti, DICHIARATI: un LOO
# calcolato su una parte dei dati confronta gli stessi modelli sugli stessi punti, ed e' lecito
# finche' lo si dice.
LOO_SOTTO = 20000  #@param {type:"integer"}
LOO_DRAWS = 600    #@param {type:"integer"}

def _loo_di(costruisci, chiave):
    idt = campiona(chiave, costruisci, draws=LOO_DRAWS, tune=LOO_DRAWS,
                   target_accept=0.95, loglik=True, seme=SEED+11, silenzioso=True)
    try:
        l = az.loo(idt, pointwise=True)
        k = np.asarray(l.pareto_k).ravel()
        return dict(chiave=chiave, elpd=float(l.elpd_loo), se=float(l.se),
                    k_max=float(np.nanmax(k)), quota_k=float(np.mean(k > 0.7)))
    except Exception as e:
        return dict(chiave=chiave, elpd=np.nan, se=np.nan, k_max=np.nan, quota_k=np.nan,
                    errore=f"{type(e).__name__}: {e}")
    finally:
        del idt; gc.collect()

banner("13. Confronti LOO")
_ix = np.random.default_rng(SEED+11).choice(len(A_y), min(LOO_SOTTO, len(A_y)), replace=False)
_ci, _ki, _bi = A_ci[_ix], A_ki[_ix], A_bi[_ix]
LOO = []

print(f"\nM1, seconda gamba: il termine di overlap vince il LOO?  "
      f"(sottocampione {len(_ix):,} triplette, {LOO_DRAWS} draws)")
_a1 = _loo_di(lambda: modello_A(y=A_y[_ix], ci=_ci, ki=_ki, bi=_bi, con_overlap=True),  "loo_A_con")
_a0 = _loo_di(lambda: modello_A(y=A_y[_ix], ci=_ci, ki=_ki, bi=_bi, con_overlap=False), "loo_A_senza")
LOO += [dict(confronto="M1 overlap", **_a1), dict(confronto="M1 overlap", **_a0)]

print(f"\nH2, nullo annidato: u_p migliora la predizione?")
_c1 = _loo_di(lambda: modello_C(y=C_y[_ix], ci=_ci, ki=_ki, bi=_bi, pi=C_pi[_ix], con_fase=True),  "loo_C_con")
_c0 = _loo_di(lambda: modello_C(y=C_y[_ix], ci=_ci, ki=_ki, bi=_bi, pi=C_pi[_ix], con_fase=False), "loo_C_senza")
LOO += [dict(confronto="H2 fase", **_c1), dict(confronto="H2 fase", **_c0)]

print(f"\nH3: il fit a due etichette vince contro ciascuna e contro nessuna?")
_jx = np.random.default_rng(SEED+11).choice(len(D1_y), min(LOO_SOTTO, len(D1_y)), replace=False)
for _nm, _s, _p in (("due", True, True), ("solo_S", True, False),
                    ("solo_P", False, True), ("nessuna", False, False)):
    LOO.append(dict(confronto="H3 etichette", variante=_nm,
                    **_loo_di(lambda s=_s, p=_p: modello_D1(y=D1_y[_jx], gi=D1_gi[_jx],
                                                            s=onS[_jx], p=onP[_jx],
                                                            usa_S=s, usa_P=p),
                              f"loo_D1_{_nm}")))
LOO_TAB = pd.DataFrame(LOO)
mostra(LOO_TAB.round(3))
LOO_TAB.to_csv(f"{OUT_DIR}/13_loo.csv", index=False)

def _vince(conf, chi, contro):
    s = LOO_TAB[LOO_TAB.confronto == conf].set_index("chiave")
    if chi not in s.index or any(c not in s.index for c in contro): return False, "assente"
    if not np.isfinite(s.loc[chi, "elpd"]): return False, "elpd non calcolabile"
    if s.loc[chi, "k_max"] > 0.7 or any(s.loc[c, "k_max"] > 0.7 for c in contro):
        return False, f"Pareto-k oltre 0.7 (max {s.k_max.max():.2f}): confronto inaffidabile"
    for c in contro:
        diff = s.loc[chi, "elpd"] - s.loc[c, "elpd"]
        dse = float(np.sqrt(s.loc[chi, "se"]**2 + s.loc[c, "se"]**2))
        if not (diff > 2*dse): return False, f"non separato da {c} ({diff:+.1f} vs 2*dse {2*dse:.1f})"
    return True, "vince"

M1_LOO, M1_LOO_NOTA = _vince("M1 overlap", "loo_A_con", ["loo_A_senza"])
H2_LOO, H2_LOO_NOTA = _vince("H2 fase", "loo_C_con", ["loo_C_senza"])
H3_LOO, H3_LOO_NOTA = _vince("H3 etichette", "loo_D1_due",
                             ["loo_D1_solo_S", "loo_D1_solo_P", "loo_D1_nessuna"])
print(f"\nM1, il termine di overlap vince: {M1_LOO}   ({M1_LOO_NOTA})")
print(f"H2, u_p migliora la predizione:  {H2_LOO}   ({H2_LOO_NOTA})")
print(f"H3, il fit a due etichette vince: {H3_LOO}   ({H3_LOO_NOTA})")

In [ ]:
#@title 14. Sensibilita' ai priori, la scala dichiarata {0.25, 0.5, 1.0}  { display-mode: "form" }
# "The range over which the prior scale is varied as a sensitivity check is {0.25, 0.5, 1.0};
# all are centred on no effect, so a posterior that has not moved from its prior is itself a
# report of an absent effect." E' una dichiarazione del Deliverable, quindi va eseguita.
LADDER_DRAWS = max(400, DRAWS // 2)
banner("14. Ladder dei priori")
_ix = np.random.default_rng(SEED+13).choice(len(A_y), min(LOO_SOTTO, len(A_y)), replace=False)
_ci, _ki, _bi = A_ci[_ix], A_ki[_ix], A_bi[_ix]
LAD = []
for _sc in (0.25, 0.5, 1.0):
    _id = campiona(f"lad_A_{_sc}", lambda s=_sc: modello_A(y=A_y[_ix], ci=_ci, ki=_ki, bi=_bi, scala=s),
                   draws=LADDER_DRAWS, tune=LADDER_DRAWS, target_accept=0.95,
                   metrica_densa=METRICA_DENSA, max_treedepth=MAX_TREEDEPTH,
                   seme=SEED+13, silenzioso=True)
    _b = _id.posterior["bbar"].values.ravel(); _d = _id.posterior["delta_O"].values.ravel()
    _rh, _es, _dv, _, _ = gates(_id, nomi=("bbar", "delta_O"), mostra_peggiori=False)
    LAD.append(dict(fit="A", scala=_sc, bbar=float(_b.mean()),
                    P_H1_sup=float((_b < LOG_08).mean()), P_H1_ref=float((np.abs(_b) < LOG_11).mean()),
                    P_M1_sup=float((_d > 0).mean()), rhat=_rh, ess=_es, div=_dv,
                    conv_ok=bool(_rh <= RHAT_MAX and _es >= ESS_MIN and _dv <= DIV_MAX)))
    del _id; gc.collect()
for _sc in (0.25, 0.5, 1.0):
    _id = campiona(f"lad_C_{_sc}", lambda s=_sc: modello_C(y=C_y[_ix], ci=_ci, ki=_ki, bi=_bi,
                                                           pi=C_pi[_ix], scala_phi=s*0.5),
                   draws=LADDER_DRAWS, tune=LADDER_DRAWS, target_accept=0.95,
                   metrica_densa=METRICA_DENSA, max_treedepth=MAX_TREEDEPTH,
                   seme=SEED+13, silenzioso=True)
    _s = _id.posterior["sigma_phi"].values.ravel()
    _rh, _es, _dv, _, _ = gates(_id, nomi=("sigma_phi",), mostra_peggiori=False)
    LAD.append(dict(fit="C", scala=_sc, bbar=np.nan, P_H1_sup=np.nan, P_H1_ref=np.nan,
                    P_M1_sup=np.nan, P_H2_sup=float((_s < LOG_11).mean()),
                    rhat=_rh, ess=_es, div=_dv,
                    conv_ok=bool(_rh <= RHAT_MAX and _es >= ESS_MIN and _dv <= DIV_MAX)))
    del _id; gc.collect()
LADDER = pd.DataFrame(LAD)
mostra(LADDER.round(4))
LADDER.to_csv(f"{OUT_DIR}/14_ladder.csv", index=False)

def _spread(fit, col):
    s = LADDER[(LADDER.fit == fit) & LADDER.conv_ok]
    v = s[col].dropna()
    return float(v.max() - v.min()) if len(v) >= 2 else np.nan

SENS = {}
for _f, _c, _n in (("A", "P_H1_sup", "H1 supportata"), ("A", "P_H1_ref", "H1 rifiutata"),
                   ("A", "P_M1_sup", "M1 supportata"), ("C", "P_H2_sup", "H2 supportata")):
    _sp = _spread(_f, _c)
    # fail-closed: se non ci sono almeno due varianti convergenti, la sensibilita' non e' verificata
    SENS[_n] = bool(np.isfinite(_sp) and _sp <= 0.10)
    print(f"  {_n:16s} escursione della probabilita' sulla scala: "
          + (f"{_sp:.3f}" if np.isfinite(_sp) else "non calcolabile")
          + f"   -> {'robusta' if SENS[_n] else 'FRAGILE o non verificata'}")

In [ ]:
#@title 15. Verdetti fail-closed  { display-mode: "form" }
# Un verdetto si riporta solo se TUTTI i controlli che il Deliverable ha dichiarato sono stati
# eseguiti e passati. Un controllo mancante conta come fallito: e' l'unico modo perche' la
# tabella non prometta piu' di quanto e' stato verificato.
banner("15. Verdetti")

def _conv(nome):
    s = DIAG[DIAG.fit == nome]
    return bool(len(s) and s.gate_ok.iloc[0])

CANCELLI = {
    "H1 comportamentale": dict(
        conv=_conv("A"), ppc=ppc_ok("A"),
        recupero=recupero_ok("Student-t gerarchica", ["bbar"]),
        sensibilita=SENS.get("H1 supportata", False) and SENS.get("H1 rifiutata", False)),
    "H1 rappresentazionale": dict(
        conv=_conv("B"), ppc=ppc_ok("B"),
        recupero=recupero_ok("Gamma+log", ["theta_lock"]),
        sensibilita=True),
    "H2": dict(
        conv=_conv("C"), ppc=ppc_ok("C"),
        recupero=recupero_ok("Student-t gerarchica", ["bbar"]),
        sensibilita=SENS.get("H2 supportata", False), loo=H2_LOO),
    "H3 ramo stride": dict(conv=_conv("D1"), ppc=ppc_ok("D1"),
        recupero=recupero_ok("Normale+etichette", ["theta_S"]), sensibilita=True, loo=H3_LOO),
    "H3 ramo patch": dict(conv=_conv("D1"), ppc=ppc_ok("D1"),
        recupero=recupero_ok("Normale+etichette", ["theta_P"]), sensibilita=True, loo=H3_LOO),
    "M1 mitigazione": dict(conv=_conv("A"), ppc=ppc_ok("A"),
        recupero=recupero_ok("Student-t gerarchica", ["delta_O"]),
        sensibilita=SENS.get("M1 supportata", False), loo=M1_LOO),
}

VERD = pd.DataFrame(RIGHE_VERDETTO + [M1_riga])
def _finale(r):
    g = CANCELLI.get(r.ipotesi)
    if r.esito in ("NON IDENTIFICATO",): return r.esito, ""
    if g is None: return r.esito, ""
    falliti = [k for k, v in g.items() if not v]
    if falliti: return "NON RIPORTABILE", ";".join(falliti)
    return r.esito, ""
VERD[["verdetto", "gate_falliti"]] = VERD.apply(
    lambda r: pd.Series(_finale(r)), axis=1)

# H3 e' un gate CONGIUNTO nel Deliverable: due etichette che vincono il LOO e entrambe negative
_s = VERD[VERD.modello == "D1 ramo S"]; _p = VERD[VERD.modello == "D1 ramo P"]
if len(_s) and len(_p):
    _ok = bool(H3_LOO and _s.p_sup.iloc[0] >= SOGLIA and _p.p_sup.iloc[0] >= SOGLIA)
    _cong = ("SUPPORTATA" if _ok else
             "RIFIUTATA" if (_s.p_ref.iloc[0] >= SOGLIA or _p.p_ref.iloc[0] >= SOGLIA)
             else "INCONCLUSIVA")
    print(f"H3, gate congiunto del Deliverable (LOO {H3_LOO}, "
          f"Pr(theta_S<0)={_s.p_sup.iloc[0]:.3f}, Pr(theta_P<0)={_p.p_sup.iloc[0]:.3f})"
          f"  ->  {_cong}\n")

_col = ["modello", "ipotesi", "estimando", "stima", "lo", "hi", "p_sup", "p_ref",
        "rhat", "ess", "div", "esito", "verdetto", "gate_falliti"]
VERD = VERD[[c for c in _col if c in VERD.columns]]
mostra(VERD.round(4))
VERD.to_csv(f"{OUT_DIR}/15_verdetti.csv", index=False)

print("\nLegenda dei gate: conv = convergenza, ppc = controlli predittivi, recupero = parametri")
print("ritrovati su dati sintetici, sensibilita' = la conclusione non si muove sulla scala dei")
print("priori, loo = il confronto che la regola del Deliverable richiede.")
print("\nUn gate mancante conta come fallito. La tabella non promette piu' di quanto e' verificato.")
print(f"\nscritti in {OUT_DIR}/: 10_convergenza.csv, 11_ppc.csv, 12_recupero.csv,")
print("                       13_loo.csv, 14_ladder.csv, 15_verdetti.csv, ckpt/*.nc")